# Error Analysis: Where Does the Model Lose Bits?

Systematic investigation of per-token, per-position, per-document, and per-category
loss to identify the biggest BPB improvement opportunities.

**Sections:**
1. Per-token loss collection (the big scan)
2. Token-level analysis (hardest tokens, frequency, byte-length, category)
3. Positional / structural analysis (sequence position, document position, word boundaries)
4. Document-level analysis (per-document BPB, hardest/easiest docs)
5. Neural model vs n-gram gap
6. Confidence / calibration
7. Context / repetition
8. Surprise analysis (highest individual losses)
9. Actionable BPB decomposition

In [ ]:
# === CONFIGURATION ===
RUN_ID = "baseline_20k_iters_seq2_20260324_205335"
USE_QUANTIZED = True
SEQ_LEN = 1024
DATA_PATH = "./data/datasets/fineweb10B_sp1024"
TOKENIZER_PATH = "./data/tokenizers/fineweb_1024_bpe.model"

# Analysis parameters
N_BATCHES = 500  # number of validation sequences to scan (~512K tokens)
N_NGRAM_BATCHES = 100  # sequences for KenLM comparison (slower)

In [ ]:
import glob
import importlib.util
import io
import math
import os
import re
import sys
import tempfile
import zlib
from collections import Counter
from pathlib import Path

PLOT_DIR = Path("plots/explore_model_errors")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

plt.rcParams["figure.dpi"] = 120
ln2 = math.log(2)

In [ ]:
# --- Load training script from log and import as module ---
run_dir = Path("models") / RUN_ID
log_txt = run_dir / "log.txt"
assert log_txt.exists(), f"Log file not found: {log_txt}"

if USE_QUANTIZED:
    ckpt_path = run_dir / "model.int8.ptz"
else:
    ckpt_path = run_dir / "model.npz"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

raw_lines = log_txt.read_text().splitlines()

in_main = False
source_end = len(raw_lines)
for i, line in enumerate(raw_lines):
    if line.strip().startswith("if __name__"):
        in_main = True
    if in_main and i > 0:
        stripped = line.strip()
        if stripped and not stripped.startswith("#") and ":" in stripped:
            first_token = stripped.split(":")[0].split("(")[0].split("[")[0]
            if first_token in (
                "step",
                "val_progress",
                "saved_model",
                "serialized_model_int8_zlib",
                "final_int8_zlib_roundtrip",
                "final_int8_zlib_roundtrip_exact",
                "stopping_early",
                "WARNING",
            ):
                source_end = i
                break

main_idx = None
for i, line in enumerate(raw_lines[:source_end]):
    if line.strip().startswith("if __name__"):
        main_idx = i
        break

importable_source = (
    "\n".join(raw_lines[:main_idx]) if main_idx else "\n".join(raw_lines[:source_end])
)

_tmpdir = tempfile.mkdtemp()
_tmp_path = os.path.join(_tmpdir, f"{RUN_ID.replace('-', '_')}_module.py")
with open(_tmp_path, "w") as f:
    f.write(importable_source)

spec = importlib.util.spec_from_file_location("train_module", _tmp_path)
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)
print(f"Imported training module from {log_txt}")

In [ ]:
# --- Build model and load weights ---
import inspect

hp = train_module.Hyperparameters()
GPT = train_module.GPT
sig = inspect.signature(GPT.__init__)
init_params = [p for p in sig.parameters if p != "self"]

param_map = {
    "vocab_size": hp.vocab_size,
    "num_layers": hp.num_layers,
    "dim": hp.model_dim,
    "num_heads": hp.num_heads,
    "num_kv_heads": hp.num_kv_heads,
    "mlp_mult": hp.mlp_mult,
    "logit_chunk_tokens": getattr(hp, "logit_chunk_tokens", 0),
    "logit_softcap": getattr(hp, "logit_softcap", 30.0),
    "rope_base": getattr(hp, "rope_base", 10000.0),
    "tied_embed_init_std": getattr(hp, "tied_embed_init_std", 0.005),
    "qk_gain_init": getattr(hp, "qk_gain_init", 1.5),
}

kwargs = {}
for p in init_params:
    if p in param_map:
        kwargs[p] = param_map[p]
    elif hasattr(hp, p):
        kwargs[p] = getattr(hp, p)

model = GPT(**kwargs)

if USE_QUANTIZED:
    with open(ckpt_path, "rb") as f:
        quant_blob = f.read()
    quant_obj = torch.load(io.BytesIO(zlib.decompress(quant_blob)), map_location="cpu")
    state_dict = train_module.dequantize_state_dict_int8(quant_obj)
else:
    state_dict = torch.load(ckpt_path, map_location="cpu")

model.load_state_dict(state_dict, strict=True)
model.eval()
num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {num_params:,} parameters")

In [ ]:
# --- Load tokenizer and data ---
sp_tok = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
V = sp_tok.vocab_size()
print(f"Tokenizer: vocab_size={V}")

sys.path.insert(0, str(Path(".").resolve()))
from train_gpt import load_validation_tokens, TokenStream, build_sentencepiece_luts

val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"
val_tokens = load_validation_tokens(val_pattern, seq_len=SEQ_LEN)
print(
    f"Validation tokens: {val_tokens.shape[0]:,} ({(val_tokens.shape[0] - 1) // SEQ_LEN} sequences)"
)

train_pattern = f"{DATA_PATH}/fineweb_train_*.bin"
train_stream = TokenStream(train_pattern)

base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(
    sp_tok, hp.vocab_size, torch.device("cpu")
)

In [ ]:
# --- Helper functions ---
def get_val_batch(batch_idx: int = 0, seq_len: int = SEQ_LEN):
    start = batch_idx * seq_len
    end = start + seq_len + 1
    if end > val_tokens.shape[0]:
        raise IndexError(f"batch_idx {batch_idx} out of range")
    chunk = val_tokens[start:end]
    return chunk[:-1].reshape(1, seq_len).long(), chunk[1:].reshape(1, seq_len).long()


def decode_tokens(token_ids):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()
    if isinstance(token_ids, np.ndarray):
        token_ids = token_ids.tolist()
    if isinstance(token_ids[0], list):
        token_ids = token_ids[0]
    return sp_tok.decode(token_ids)


@torch.no_grad()
def get_logits(x):
    """Run forward pass and return logits (before loss reduction)."""
    emb = model.tok_emb(x)
    h = F.rms_norm(emb, (emb.size(-1),))
    x0 = h
    skips = []
    for i in range(model.num_encoder_layers):
        h = model.blocks[i](h, x0)
        skips.append(h)
    for i in range(model.num_decoder_layers):
        if skips:
            h = h + model.skip_weights[i].to(dtype=h.dtype)[None, None, :] * skips.pop()
        h = model.blocks[model.num_encoder_layers + i](h, x0)
    h = model.final_norm(h).reshape(-1, h.size(-1))
    if model.tie_embeddings:
        logits = F.linear(h, model.tok_emb.weight)
    else:
        logits = model.lm_head(h)
    return model.logit_softcap * torch.tanh(logits / model.logit_softcap)


print("Helpers defined: get_val_batch, decode_tokens, get_logits")

In [ ]:
# --- Build token metadata table ---
id_to_piece = [sp_tok.id_to_piece(i) for i in range(V)]
piece_to_id = {p: i for i, p in enumerate(id_to_piece)}


def classify_token(tok_id):
    """Classify a token into a category."""
    if tok_id == 1:
        return "bos"
    piece = id_to_piece[tok_id]
    if piece.startswith("<0x") and piece.endswith(">"):
        return "byte_fallback"
    if piece.startswith("\u2581"):  # SentencePiece leading space
        rest = piece[1:]
        if not rest:
            return "space_only"
        if rest.replace(",", "").replace(".", "").isdigit():
            return "number"
        if rest[0].isupper():
            return "uppercase_start"
        return "space_word"
    if all(not c.isalnum() for c in piece):
        return "punctuation"
    return "word_internal"


token_meta = pd.DataFrame(
    {
        "token_id": range(V),
        "piece": id_to_piece,
        "byte_len": base_bytes_lut.numpy(),
        "has_space": has_leading_space_lut.numpy().astype(bool),
        "is_boundary": is_boundary_token_lut.numpy().astype(bool),
        "category": [classify_token(i) for i in range(V)],
    }
)

# Token frequency from training data
N_FREQ_TOKENS = 10_000_000
freq_tok_ids = train_stream.take(N_FREQ_TOKENS).numpy()
train_freq = np.bincount(freq_tok_ids, minlength=V).astype(np.float64)
train_freq /= train_freq.sum()  # normalize to probability
token_meta["train_freq"] = train_freq

print("Token metadata:")
print(token_meta["category"].value_counts().to_string())
print(f"\nTotal vocab: {V}")

## 1. Per-Token Loss Collection

Compute per-position cross-entropy, entropy, and top-1 prediction for many validation sequences.
This DataFrame is the foundation for all subsequent analysis.

In [ ]:
# --- Precompute document positions in validation tokens ---
# For each token position, compute distance from the most recent BOS (id=1)
bos_positions = np.where(val_tokens.numpy() == 1)[0]
doc_pos_array = np.zeros(len(val_tokens), dtype=np.int32)
current_doc_start = 0
for bp in bos_positions:
    doc_pos_array[bp] = 0
    current_doc_start = bp
# Fill in distances from BOS
prev_bos = 0
for bp in np.append(bos_positions, len(val_tokens)):
    doc_pos_array[prev_bos:bp] = np.arange(bp - prev_bos)
    prev_bos = bp

print(f"Doc positions computed. BOS count: {len(bos_positions):,}")

In [ ]:
# --- Collect per-token losses ---
all_token_ids = []
all_prev_ids = []
all_losses = []
all_entropies = []
all_top1_preds = []
all_positions = []
all_doc_positions = []
all_seq_idx = []

for batch_idx in tqdm(range(N_BATCHES), desc="Scanning val sequences"):
    x, y = get_val_batch(batch_idx)
    logits = get_logits(x)  # (SEQ_LEN, V)
    log_probs = F.log_softmax(logits.float(), dim=-1)
    probs = log_probs.exp()

    target_ids = y[0]  # (SEQ_LEN,)
    input_ids = x[0]  # (SEQ_LEN,)

    # Per-token NLL
    per_token_nll = -log_probs[torch.arange(SEQ_LEN), target_ids].cpu().numpy()
    # Model entropy
    entropy = -(probs * log_probs).sum(dim=-1).cpu().numpy()
    # Top-1 prediction
    top1 = logits.argmax(dim=-1).cpu().numpy()

    # Global positions for doc_position lookup
    global_start = batch_idx * SEQ_LEN
    # Target tokens are at positions global_start+1 .. global_start+SEQ_LEN
    target_global_positions = np.arange(global_start + 1, global_start + SEQ_LEN + 1)

    all_token_ids.append(target_ids.cpu().numpy())
    all_prev_ids.append(input_ids.cpu().numpy())
    all_losses.append(per_token_nll)
    all_entropies.append(entropy)
    all_top1_preds.append(top1)
    all_positions.append(np.arange(SEQ_LEN))
    all_doc_positions.append(doc_pos_array[target_global_positions])
    all_seq_idx.append(np.full(SEQ_LEN, batch_idx))

# Concatenate
all_token_ids = np.concatenate(all_token_ids)
all_prev_ids = np.concatenate(all_prev_ids)
all_losses = np.concatenate(all_losses)
all_entropies = np.concatenate(all_entropies)
all_top1_preds = np.concatenate(all_top1_preds)
all_positions = np.concatenate(all_positions)
all_doc_positions = np.concatenate(all_doc_positions)
all_seq_idx = np.concatenate(all_seq_idx)

print(f"Collected {len(all_losses):,} token-level observations")
print(f"Mean loss: {all_losses.mean():.4f} nats")
print(f"Mean loss: {all_losses.mean() / ln2:.4f} bits")

In [ ]:
# --- Build analysis DataFrame ---
# Compute per-token byte count (for BPB) using the same formula as eval_val
token_bytes = base_bytes_lut.numpy()[all_token_ids].astype(np.float64)
# Add leading space byte when token has leading space AND previous token is not a boundary
leading_space_bytes = (
    has_leading_space_lut.numpy()[all_token_ids].astype(bool)
    & ~is_boundary_token_lut.numpy()[all_prev_ids].astype(bool)
).astype(np.float64)
token_bytes += leading_space_bytes

df = pd.DataFrame(
    {
        "token_id": all_token_ids,
        "prev_id": all_prev_ids,
        "loss": all_losses,
        "entropy": all_entropies,
        "top1_pred": all_top1_preds,
        "position": all_positions,
        "doc_position": all_doc_positions,
        "seq_idx": all_seq_idx,
        "bytes": token_bytes,
    }
)

# Per-token BPB
df["bpb"] = (df["loss"] / ln2) / df["bytes"].clip(lower=1)
df["is_correct"] = df["top1_pred"] == df["token_id"]

# Join token metadata
df["piece"] = df["token_id"].map(lambda x: id_to_piece[x])
df["category"] = df["token_id"].map(lambda x: token_meta.loc[x, "category"])
df["byte_len"] = df["token_id"].map(lambda x: token_meta.loc[x, "byte_len"])
df["has_space"] = df["token_id"].map(lambda x: token_meta.loc[x, "has_space"])
df["train_freq"] = df["token_id"].map(lambda x: token_meta.loc[x, "train_freq"])
df["log_freq"] = np.log10(df["train_freq"].clip(lower=1e-10))

# Sanity check: overall BPB
total_bits = df["loss"].sum() / ln2
total_bytes = df["bytes"].sum()
overall_bpb = total_bits / total_bytes
print(f"Overall BPB: {overall_bpb:.4f}")
print(f"Total tokens: {len(df):,}, total bytes: {total_bytes:,.0f}")
print(f"Top-1 accuracy: {df['is_correct'].mean():.3%}")
print(f"\nDataFrame shape: {df.shape}")
df.describe()

## 2. Token-Level Analysis

Which tokens are hardest? How does difficulty relate to frequency, byte-length, and category?

In [ ]:
# --- Hardest tokens: by mean loss and by total loss contribution ---
MIN_COUNT = 10

token_stats = (
    df.groupby("token_id")
    .agg(
        mean_loss=("loss", "mean"),
        mean_bpb=("bpb", "mean"),
        count=("loss", "count"),
        total_loss=("loss", "sum"),
        total_bytes=("bytes", "sum"),
    )
    .reset_index()
)
token_stats["piece"] = token_stats["token_id"].map(lambda x: id_to_piece[x])
token_stats["category"] = token_stats["token_id"].map(
    lambda x: token_meta.loc[x, "category"]
)
token_stats["train_freq"] = token_stats["token_id"].map(
    lambda x: token_meta.loc[x, "train_freq"]
)
token_stats["bpb_contribution"] = (token_stats["total_loss"] / ln2) / total_bytes

# Filter for tokens with enough observations
ts_filtered = token_stats[token_stats["count"] >= MIN_COUNT]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 by mean loss
top_mean = ts_filtered.nlargest(20, "mean_loss")
axes[0].barh(range(20), top_mean["mean_loss"].values, color="salmon")
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(
    [
        f"{r['piece']!r} ({r['category']}, n={r['count']})"
        for _, r in top_mean.iterrows()
    ],
    fontsize=8,
)
axes[0].set_xlabel("Mean Loss (nats)")
axes[0].set_title(f"Top 20 Tokens by Mean Loss (min count={MIN_COUNT})")
axes[0].invert_yaxis()

# Top 20 by total loss contribution (BPB)
top_total = token_stats.nlargest(20, "bpb_contribution")
axes[1].barh(range(20), top_total["bpb_contribution"].values, color="steelblue")
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(
    [
        f"{r['piece']!r} ({r['category']}, n={r['count']})"
        for _, r in top_total.iterrows()
    ],
    fontsize=8,
)
axes[1].set_xlabel("BPB Contribution")
axes[1].set_title("Top 20 Tokens by Total BPB Contribution")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Print table of top contributors
print("\nTop 20 tokens by total BPB contribution:")
print(
    top_total[
        ["piece", "category", "mean_loss", "mean_bpb", "count", "bpb_contribution"]
    ].to_string(index=False)
)

In [ ]:
# --- Loss vs token frequency ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ts = ts_filtered.copy()
cat_colors = {
    "space_word": "C0",
    "word_internal": "C1",
    "uppercase_start": "C2",
    "number": "C3",
    "punctuation": "C4",
    "bos": "C5",
    "byte_fallback": "C6",
    "space_only": "C7",
    "other": "C8",
}

for cat, group in ts.groupby("category"):
    color = cat_colors.get(cat, "gray")
    axes[0].scatter(
        np.log10(group["train_freq"].clip(lower=1e-10)),
        group["mean_loss"],
        alpha=0.4,
        s=10,
        label=cat,
        color=color,
    )
axes[0].set_xlabel("log10(train frequency)")
axes[0].set_ylabel("Mean Loss (nats)")
axes[0].set_title("Mean Loss vs Token Frequency")
axes[0].legend(fontsize=7, markerscale=2)

# Binned mean for total contribution
for cat, group in ts.groupby("category"):
    color = cat_colors.get(cat, "gray")
    axes[1].scatter(
        np.log10(group["train_freq"].clip(lower=1e-10)),
        group["bpb_contribution"],
        alpha=0.4,
        s=10,
        label=cat,
        color=color,
    )
axes[1].set_xlabel("log10(train frequency)")
axes[1].set_ylabel("BPB Contribution")
axes[1].set_title("BPB Contribution vs Token Frequency")
axes[1].legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.show()

In [ ]:
# --- Loss by token byte-length ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

byte_len_groups = df.groupby("byte_len")
bl_stats = byte_len_groups.agg(
    mean_loss=("loss", "mean"),
    mean_bpb=("bpb", "mean"),
    count=("loss", "count"),
    total_loss=("loss", "sum"),
    total_bytes=("bytes", "sum"),
)
bl_stats["actual_bpb"] = (bl_stats["total_loss"] / ln2) / bl_stats["total_bytes"]

# Box plot of per-token loss by byte length
data_by_bl = [group["loss"].values for _, group in byte_len_groups]
labels = [str(bl) for bl in byte_len_groups.groups.keys()]
bp = axes[0].boxplot(data_by_bl, labels=labels, showfliers=False, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("lightblue")
axes[0].set_xlabel("Token Byte Length")
axes[0].set_ylabel("Loss (nats)")
axes[0].set_title("Loss Distribution by Token Byte Length")

# BPB by byte length
axes[1].bar(bl_stats.index, bl_stats["actual_bpb"], color="steelblue")
axes[1].set_xlabel("Token Byte Length")
axes[1].set_ylabel("BPB (bits/byte)")
axes[1].set_title("BPB by Token Byte Length")
for i, (bl, row) in enumerate(bl_stats.iterrows()):
    axes[1].text(
        bl, row["actual_bpb"] + 0.02, f"n={row['count']:,}", ha="center", fontsize=7
    )

plt.tight_layout()
plt.show()

In [ ]:
# --- Loss by token category (the key decomposition) ---
cat_stats = df.groupby("category").agg(
    mean_loss=("loss", "mean"),
    mean_bpb=("bpb", "mean"),
    count=("loss", "count"),
    total_loss=("loss", "sum"),
    total_bytes=("bytes", "sum"),
    mean_entropy=("entropy", "mean"),
    accuracy=("is_correct", "mean"),
)
cat_stats["actual_bpb"] = (cat_stats["total_loss"] / ln2) / cat_stats["total_bytes"]
cat_stats["frac_tokens"] = cat_stats["count"] / cat_stats["count"].sum()
cat_stats["frac_bytes"] = cat_stats["total_bytes"] / cat_stats["total_bytes"].sum()
cat_stats["bpb_contribution"] = (cat_stats["total_loss"] / ln2) / total_bytes
cat_stats["frac_bpb"] = cat_stats["bpb_contribution"] / overall_bpb
cat_stats = cat_stats.sort_values("bpb_contribution", ascending=False)

print("=== Loss by Token Category ===")
print(
    cat_stats[
        [
            "mean_loss",
            "actual_bpb",
            "frac_tokens",
            "frac_bytes",
            "bpb_contribution",
            "frac_bpb",
            "accuracy",
        ]
    ].to_string(float_format="%.4f")
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean loss and BPB by category
cats = cat_stats.index.tolist()
x = np.arange(len(cats))
axes[0].bar(
    x - 0.2, cat_stats["mean_loss"], 0.4, label="Mean Loss (nats)", color="salmon"
)
axes[0].bar(x + 0.2, cat_stats["actual_bpb"], 0.4, label="BPB", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
axes[0].legend()
axes[0].set_title("Mean Loss / BPB by Category")

# Stacked BPB contribution
colors = plt.cm.Set2(np.linspace(0, 1, len(cats)))
bottom = 0
for i, cat in enumerate(cats):
    val = cat_stats.loc[cat, "bpb_contribution"]
    axes[1].bar(0, val, bottom=bottom, color=colors[i], label=f"{cat} ({val:.4f})")
    bottom += val
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylabel("BPB")
axes[1].set_title(f"BPB Decomposition (total={overall_bpb:.4f})")
axes[1].legend(fontsize=7, loc="center right")
axes[1].set_xticks([])

# Token fraction vs BPB fraction
axes[2].bar(
    x - 0.2, cat_stats["frac_tokens"], 0.4, label="% tokens", color="lightgreen"
)
axes[2].bar(x + 0.2, cat_stats["frac_bpb"], 0.4, label="% BPB", color="coral")
axes[2].set_xticks(x)
axes[2].set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
axes[2].legend()
axes[2].set_title("Token Fraction vs BPB Fraction")

plt.tight_layout()
plt.show()

## 3. Positional / Structural Analysis

How does loss vary with position in sequence, position in document, and local context?

In [ ]:
# --- Loss vs position in sequence ---
pos_stats = df.groupby("position").agg(
    mean_loss=("loss", "mean"),
    frac_bos=("token_id", lambda x: (x == 1).mean()),
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(pos_stats.index, pos_stats["mean_loss"], color="steelblue", linewidth=0.8)
ax1.set_xlabel("Position in Sequence")
ax1.set_ylabel("Mean Loss (nats)", color="steelblue")
ax1.set_title("Mean Loss vs Position in Sequence")

ax2 = ax1.twinx()
ax2.plot(
    pos_stats.index, pos_stats["frac_bos"], color="coral", alpha=0.5, linewidth=0.8
)
ax2.set_ylabel("Fraction BOS tokens", color="coral")

plt.tight_layout()
plt.show()

In [ ]:
# --- Loss vs position in document (warm-up curve) ---
# Filter to doc positions with enough samples
doc_pos_stats = df.groupby("doc_position").agg(
    mean_loss=("loss", "mean"),
    mean_bpb=("bpb", "mean"),
    count=("loss", "count"),
)
doc_pos_stats = doc_pos_stats[doc_pos_stats["count"] >= 50]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Zoom: first 100 positions (warm-up)
dp_zoom = doc_pos_stats[doc_pos_stats.index <= 100]
axes[0].plot(dp_zoom.index, dp_zoom["mean_loss"], color="steelblue", linewidth=1.5)
axes[0].axhline(
    df["loss"].mean(),
    color="gray",
    linestyle="--",
    alpha=0.5,
    label=f"Overall mean: {df['loss'].mean():.3f}",
)
axes[0].set_xlabel("Position in Document")
axes[0].set_ylabel("Mean Loss (nats)")
axes[0].set_title("Document Warm-up: Loss vs Position (first 100)")
axes[0].legend()

# Full range
axes[1].plot(
    doc_pos_stats.index, doc_pos_stats["mean_loss"], color="steelblue", linewidth=0.8
)
axes[1].axhline(df["loss"].mean(), color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Position in Document")
axes[1].set_ylabel("Mean Loss (nats)")
axes[1].set_title("Loss vs Position in Document (full range)")

plt.tight_layout()
plt.show()

# Print key statistics
for dp in [0, 1, 2, 3, 5, 10, 20, 50]:
    if dp in doc_pos_stats.index:
        row = doc_pos_stats.loc[dp]
        print(
            f"  doc_pos={dp:3d}: mean_loss={row['mean_loss']:.3f}, mean_bpb={row['mean_bpb']:.3f}, n={row['count']:,}"
        )

In [ ]:
# --- Word-boundary vs word-internal loss ---
df_word = df[
    df["category"].isin(["space_word", "uppercase_start", "word_internal", "number"])
].copy()
df_word["is_word_start"] = df_word["has_space"]

ws_stats = df_word.groupby("is_word_start").agg(
    mean_loss=("loss", "mean"),
    mean_bpb=("bpb", "mean"),
    count=("loss", "count"),
    total_loss=("loss", "sum"),
    total_bytes=("bytes", "sum"),
)
ws_stats["actual_bpb"] = (ws_stats["total_loss"] / ln2) / ws_stats["total_bytes"]
print("Word-start vs word-internal:")
print(ws_stats.to_string(float_format="%.4f"))
print(
    f"\nWord prediction gap: {ws_stats.loc[True, 'mean_loss'] - ws_stats.loc[False, 'mean_loss']:.3f} nats"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograms
for label, mask, color in [
    ("Word-start", True, "coral"),
    ("Word-internal", False, "steelblue"),
]:
    subset = df_word[df_word["is_word_start"] == mask]
    axes[0].hist(
        subset["loss"],
        bins=100,
        alpha=0.6,
        label=f"{label} (n={len(subset):,})",
        color=color,
        density=True,
    )
axes[0].set_xlabel("Loss (nats)")
axes[0].set_ylabel("Density")
axes[0].set_title("Loss Distribution: Word-start vs Word-internal")
axes[0].legend()
axes[0].set_xlim(0, 12)

# CDF
for label, mask, color in [
    ("Word-start", True, "coral"),
    ("Word-internal", False, "steelblue"),
]:
    subset = df_word[df_word["is_word_start"] == mask]["loss"].sort_values()
    axes[1].plot(subset, np.linspace(0, 1, len(subset)), label=label, color=color)
axes[1].set_xlabel("Loss (nats)")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF of Loss: Word-start vs Word-internal")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Loss conditioned on preceding token ---
prev_counts = df["prev_id"].value_counts()
top_prev = prev_counts.head(50).index.tolist()

prev_stats = (
    df[df["prev_id"].isin(top_prev)]
    .groupby("prev_id")
    .agg(
        mean_next_loss=("loss", "mean"),
        count=("loss", "count"),
    )
    .sort_values("mean_next_loss", ascending=False)
)
prev_stats["piece"] = prev_stats.index.map(lambda x: id_to_piece[x])

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(prev_stats)), prev_stats["mean_next_loss"].values, color="steelblue")
ax.set_yticks(range(len(prev_stats)))
ax.set_yticklabels(
    [f"{r['piece']!r} (n={r['count']:,})" for _, r in prev_stats.iterrows()], fontsize=7
)
ax.set_xlabel("Mean Loss of Next Token (nats)")
ax.set_title("Mean Next-Token Loss by Preceding Token (top 50 most common)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Document-Level Analysis

Which documents are hardest? What characterizes difficult vs easy documents?

In [ ]:
# --- Segment scanned region into documents ---
# Find BOS positions in the scanned range of val_tokens
scan_end = N_BATCHES * SEQ_LEN + 1
val_np = val_tokens[:scan_end].numpy()
bos_in_scan = np.where(val_np == 1)[0]

doc_records = []
for i in range(len(bos_in_scan)):
    doc_start = bos_in_scan[i]
    doc_end = bos_in_scan[i + 1] if i + 1 < len(bos_in_scan) else scan_end
    doc_len = doc_end - doc_start

    # Find rows in df that belong to this document
    # Target positions in val_tokens are: for seq_idx=b, position=p -> global pos = b*SEQ_LEN + p + 1
    # We use doc_position to identify: tokens with doc_position in [0, doc_len-1]
    # But simpler: just filter by global position range
    # The target token at df row is at val_tokens position = seq_idx*SEQ_LEN + position + 1
    pass

# Simpler approach: assign each df row to a document ID using the global position
df["global_pos"] = df["seq_idx"] * SEQ_LEN + df["position"] + 1
# Find which document each global_pos belongs to
doc_id_for_pos = np.searchsorted(bos_in_scan, df["global_pos"].values, side="right") - 1
df["doc_id"] = doc_id_for_pos

# Per-document stats
doc_stats = df.groupby("doc_id").agg(
    mean_loss=("loss", "mean"),
    total_loss=("loss", "sum"),
    total_bytes=("bytes", "sum"),
    count=("loss", "count"),
    mean_entropy=("entropy", "mean"),
    accuracy=("is_correct", "mean"),
)
doc_stats["bpb"] = (doc_stats["total_loss"] / ln2) / doc_stats["total_bytes"]
# Document length from BOS positions
doc_lengths_scan = np.diff(bos_in_scan, append=scan_end)
doc_stats["doc_length"] = doc_lengths_scan[doc_stats.index]
# First 50 tokens for topic identification
doc_stats["first_text"] = [
    decode_tokens(
        val_tokens[
            bos_in_scan[i] : bos_in_scan[i] + min(50, doc_lengths_scan[i])
        ].tolist()
    )
    for i in doc_stats.index
]

print(f"Documents in scanned range: {len(doc_stats)}")
print(f"Mean doc BPB: {doc_stats['bpb'].mean():.4f}")
print(f"Median doc BPB: {doc_stats['bpb'].median():.4f}")
doc_stats.describe()

In [ ]:
# --- Per-document BPB distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter to docs with enough tokens for stable estimates
doc_stats_valid = doc_stats[doc_stats["count"] >= 20]

axes[0].hist(
    doc_stats_valid["bpb"], bins=80, color="steelblue", edgecolor="black", alpha=0.7
)
axes[0].set_xlabel("Document BPB")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Per-Document BPB Distribution (n={len(doc_stats_valid)})")
for q in [0.5, 0.9, 0.95, 0.99]:
    qval = doc_stats_valid["bpb"].quantile(q)
    axes[0].axvline(qval, color="coral", linestyle="--", alpha=0.7)
    axes[0].text(
        qval,
        axes[0].get_ylim()[1] * 0.9,
        f"p{int(q * 100)}={qval:.2f}",
        rotation=90,
        va="top",
        fontsize=8,
        color="coral",
    )

# What fraction of total loss do the hardest 10% of docs contribute?
sorted_docs = doc_stats_valid.sort_values("bpb", ascending=False)
n_top10 = max(1, len(sorted_docs) // 10)
top10_loss = sorted_docs.head(n_top10)["total_loss"].sum()
total_loss_all = sorted_docs["total_loss"].sum()
print(
    f"Top 10% hardest docs ({n_top10} docs) contribute {top10_loss / total_loss_all:.1%} of total loss"
)

# Doc length vs BPB
axes[1].scatter(
    doc_stats_valid["doc_length"],
    doc_stats_valid["bpb"],
    alpha=0.3,
    s=5,
    color="steelblue",
)
axes[1].set_xlabel("Document Length (tokens)")
axes[1].set_ylabel("Document BPB")
axes[1].set_title("Document Length vs BPB")
# Binned mean
bins = pd.cut(doc_stats_valid["doc_length"], bins=20)
binned = doc_stats_valid.groupby(bins, observed=True)["bpb"].mean()
bin_centers = [(b.left + b.right) / 2 for b in binned.index]
axes[1].plot(
    bin_centers, binned.values, "r-o", markersize=4, linewidth=2, label="Binned mean"
)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Characterize hardest and easiest documents ---
print("=" * 80)
print("TOP 10 HARDEST DOCUMENTS (by BPB)")
print("=" * 80)
for rank, (idx, row) in enumerate(doc_stats_valid.nlargest(10, "bpb").iterrows()):
    print(
        f"\n--- #{rank + 1} | BPB={row['bpb']:.3f} | len={row['doc_length']} | tokens_scanned={row['count']} ---"
    )
    # Show first 200 chars of decoded text
    start = bos_in_scan[idx]
    end = start + min(200, int(row["doc_length"]))
    text = decode_tokens(val_tokens[start:end].tolist())
    print(text[:300])

print("\n" + "=" * 80)
print("TOP 10 EASIEST DOCUMENTS (by BPB)")
print("=" * 80)
for rank, (idx, row) in enumerate(doc_stats_valid.nsmallest(10, "bpb").iterrows()):
    print(
        f"\n--- #{rank + 1} | BPB={row['bpb']:.3f} | len={row['doc_length']} | tokens_scanned={row['count']} ---"
    )
    start = bos_in_scan[idx]
    end = start + min(200, int(row["doc_length"]))
    text = decode_tokens(val_tokens[start:end].tolist())
    print(text[:300])

## 5. Neural Model vs N-gram Gap Analysis

Where does the neural model add the most value over n-grams? Where does it add the least (bottlenecks)?

In [ ]:
# --- Load KenLM models ---
import kenlm

KENLM_MODEL_PATH_TEMPLATE = "./models/kenlm/kenlm_{order}gram.binary"


class KenLMWrapper:
    def __init__(self, model_path, piece_to_id, order):
        self.model = kenlm.Model(model_path)
        self.piece_to_id = piece_to_id
        self.order = order

    def _to_id_str(self, piece):
        return str(self.piece_to_id.get(piece, 0))

    def logscore(self, word, context=None):
        word_id = self._to_id_str(word)
        if context:
            ctx_ids = [self._to_id_str(p) for p in context]
            sentence = " ".join(ctx_ids + [word_id])
        else:
            sentence = word_id
        scores = list(self.model.full_scores(sentence, bos=False, eos=False))
        log10_prob = scores[-1][0]
        return log10_prob / math.log10(2)  # log10 -> log2


kenlm_models = {}
for order in range(2, 5):
    kenlm_path = KENLM_MODEL_PATH_TEMPLATE.format(order=order)
    if Path(kenlm_path).exists():
        kenlm_models[order] = KenLMWrapper(kenlm_path, piece_to_id, order)
        print(f"Loaded KenLM {order}-gram from {kenlm_path}")
    else:
        print(f"KenLM {order}-gram not found: {kenlm_path}")

best_kenlm_order = max(kenlm_models.keys()) if kenlm_models else None
print(f"Best available KenLM order: {best_kenlm_order}")

In [ ]:
# --- Compute KenLM per-token losses for a subset ---
if best_kenlm_order:
    kenlm_model = kenlm_models[best_kenlm_order]
    kenlm_losses = []

    for batch_idx in tqdm(
        range(N_NGRAM_BATCHES), desc=f"KenLM {best_kenlm_order}-gram scoring"
    ):
        x, y = get_val_batch(batch_idx)
        tokens = torch.cat([x[0, :1], y[0]], dim=0).tolist()
        pieces = [id_to_piece[t] for t in tokens]

        batch_kenlm = []
        for pos in range(SEQ_LEN):
            ctx_start = max(0, pos + 1 - best_kenlm_order + 1)
            context = pieces[ctx_start : pos + 1]
            word = pieces[pos + 1]
            log2_prob = kenlm_model.logscore(word, context if context else None)
            nll_nats = -log2_prob * ln2
            batch_kenlm.append(nll_nats)
        kenlm_losses.append(batch_kenlm)

    kenlm_losses = np.array(kenlm_losses).flatten()

    # Add to DataFrame (only for the first N_NGRAM_BATCHES * SEQ_LEN rows)
    n_ngram_rows = N_NGRAM_BATCHES * SEQ_LEN
    df_ngram = df.iloc[:n_ngram_rows].copy()
    df_ngram["kenlm_loss"] = kenlm_losses
    df_ngram["neural_advantage"] = df_ngram["kenlm_loss"] - df_ngram["loss"]

    print(
        f"KenLM mean loss: {kenlm_losses.mean():.4f} nats ({kenlm_losses.mean() / ln2:.4f} bits)"
    )
    print(
        f"Neural mean loss: {df_ngram['loss'].mean():.4f} nats ({df_ngram['loss'].mean() / ln2:.4f} bits)"
    )
    print(f"Mean advantage: {df_ngram['neural_advantage'].mean():.4f} nats")
else:
    print("No KenLM models available, skipping n-gram comparison")
    df_ngram = None

In [ ]:
# --- Gap analysis by token category and document position ---
if df_ngram is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # By category
    gap_by_cat = (
        df_ngram.groupby("category")
        .agg(
            neural_loss=("loss", "mean"),
            kenlm_loss=("kenlm_loss", "mean"),
            advantage=("neural_advantage", "mean"),
            count=("loss", "count"),
        )
        .sort_values("advantage", ascending=True)
    )

    cats = gap_by_cat.index.tolist()
    x = np.arange(len(cats))
    w = 0.35
    axes[0].bar(
        x - w / 2, gap_by_cat["neural_loss"], w, label="Neural", color="steelblue"
    )
    axes[0].bar(
        x + w / 2,
        gap_by_cat["kenlm_loss"],
        w,
        label=f"KenLM-{best_kenlm_order}",
        color="salmon",
    )
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel("Mean Loss (nats)")
    axes[0].set_title("Neural vs KenLM by Category")
    axes[0].legend()
    # Annotate advantage
    for i, cat in enumerate(cats):
        adv = gap_by_cat.loc[cat, "advantage"]
        axes[0].text(
            i,
            gap_by_cat.loc[cat, "kenlm_loss"] + 0.05,
            f"+{adv:.2f}",
            ha="center",
            fontsize=7,
            color="green" if adv > 0 else "red",
        )

    # By document position
    dp_gap = df_ngram.groupby("doc_position").agg(
        neural_loss=("loss", "mean"),
        kenlm_loss=("kenlm_loss", "mean"),
        count=("loss", "count"),
    )
    dp_gap = dp_gap[dp_gap["count"] >= 20]
    dp_gap = dp_gap[dp_gap.index <= 200]

    axes[1].plot(dp_gap.index, dp_gap["neural_loss"], label="Neural", color="steelblue")
    axes[1].plot(
        dp_gap.index,
        dp_gap["kenlm_loss"],
        label=f"KenLM-{best_kenlm_order}",
        color="salmon",
    )
    axes[1].fill_between(
        dp_gap.index,
        dp_gap["neural_loss"],
        dp_gap["kenlm_loss"],
        alpha=0.2,
        color="green",
        label="Advantage",
    )
    axes[1].set_xlabel("Position in Document")
    axes[1].set_ylabel("Mean Loss (nats)")
    axes[1].set_title("Neural vs KenLM by Document Position")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Tokens where KenLM is competitive or wins
    close_mask = df_ngram["neural_advantage"] < 0.1
    loses_mask = df_ngram["neural_advantage"] < 0
    print(
        f"\nTokens where neural advantage < 0.1 nats: {close_mask.sum():,} ({close_mask.mean():.1%})"
    )
    print(
        f"Tokens where KenLM wins (advantage < 0): {loses_mask.sum():,} ({loses_mask.mean():.1%})"
    )

    if loses_mask.any():
        print(f"\nCategory breakdown where KenLM wins:")
        print(df_ngram[loses_mask]["category"].value_counts().to_string())

## 6. Confidence / Calibration Analysis

Is the model well-calibrated? Where is it overconfident (confident but wrong) or underconfident (uncertain but right)?

In [ ]:
# --- Entropy vs loss scatter ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Hexbin density plot
hb = axes[0].hexbin(
    df["entropy"], df["loss"], gridsize=80, cmap="viridis", mincnt=1, bins="log"
)
axes[0].plot([0, 8], [0, 8], "r--", alpha=0.5, label="entropy = loss")
axes[0].set_xlabel("Model Entropy (nats)")
axes[0].set_ylabel("Actual Loss (nats)")
axes[0].set_title("Model Entropy vs Actual Loss")
axes[0].legend()
axes[0].set_xlim(0, 8)
axes[0].set_ylim(0, 12)
plt.colorbar(hb, ax=axes[0], label="log10(count)")

# By category
for cat in ["space_word", "word_internal", "uppercase_start", "punctuation", "number"]:
    subset = df[df["category"] == cat]
    axes[1].scatter(
        subset["entropy"].values[::10],
        subset["loss"].values[::10],
        alpha=0.1,
        s=2,
        label=cat,
    )
axes[1].plot([0, 8], [0, 8], "r--", alpha=0.5)
axes[1].set_xlabel("Model Entropy (nats)")
axes[1].set_ylabel("Actual Loss (nats)")
axes[1].set_title("Entropy vs Loss by Category (subsampled)")
axes[1].legend(fontsize=7, markerscale=5)
axes[1].set_xlim(0, 8)
axes[1].set_ylim(0, 12)

plt.tight_layout()
plt.show()

In [ ]:
# --- Overconfident errors and underconfident correct predictions ---
median_entropy = df["entropy"].median()
p90_loss = df["loss"].quantile(0.90)
p90_entropy = df["entropy"].quantile(0.90)

overconfident = df[(df["entropy"] < median_entropy) & (df["loss"] > p90_loss)]
underconfident = df[(df["entropy"] > p90_entropy) & df["is_correct"]]

print(
    f"Overconfident errors (low entropy, high loss): {len(overconfident):,} ({len(overconfident) / len(df):.1%})"
)
print(
    f"  Total loss contribution: {overconfident['loss'].sum() / df['loss'].sum():.1%} of all loss"
)
print(f"  Category breakdown:")
print(f"  {overconfident['category'].value_counts().to_string()}")

print(
    f"\nUnderconfident correct (high entropy, correct top-1): {len(underconfident):,} ({len(underconfident) / len(df):.1%})"
)
print(f"  Mean loss: {underconfident['loss'].mean():.3f} (wasted bits from hedging)")
print(f"  Category breakdown:")
print(f"  {underconfident['category'].value_counts().head(5).to_string()}")

# Show examples of overconfident errors
print("\n--- Top 30 Overconfident Errors (sorted by loss) ---")
oc_sorted = overconfident.nlargest(30, "loss")
for _, row in oc_sorted.iterrows():
    # Get context (preceding 15 tokens)
    seq = int(row["seq_idx"])
    pos = int(row["position"])
    x, y = get_val_batch(seq)
    start_ctx = max(0, pos - 15)
    ctx_tokens = x[0, start_ctx : pos + 1].tolist()
    next_token = decode_tokens(x[0, pos + 2 : pos + 3].tolist())
    ctx_text = decode_tokens(ctx_tokens)
    target_piece = id_to_piece[int(row["token_id"])]
    pred_piece = id_to_piece[int(row["top1_pred"])]
    print(
        f"  loss={row['loss']:.2f} ent={row['entropy']:.2f} | ...{ctx_text[-50:]}[{target_piece!r}]{next_token} predicted={pred_piece!r} ({row['category']})"
    )

## 7. Context / Repetition Analysis

Does the model benefit from seeing tokens earlier in the context? How quickly does it "learn" within a document?

In [ ]:
# --- Repeated vs novel tokens in context ---
# For each target token, check if it appeared in the input sequence
seen_in_context = np.zeros(len(df), dtype=bool)

for batch_idx in tqdm(range(N_BATCHES), desc="Checking token repetition"):
    x, y = get_val_batch(batch_idx)
    input_ids = x[0].numpy()
    target_ids = y[0].numpy()
    start_idx = batch_idx * SEQ_LEN

    for pos in range(SEQ_LEN):
        if target_ids[pos] in input_ids[: pos + 1]:
            seen_in_context[start_idx + pos] = True

df["seen_in_context"] = seen_in_context

# Compare
rep_stats = df.groupby("seen_in_context").agg(
    mean_loss=("loss", "mean"),
    mean_bpb=("bpb", "mean"),
    count=("loss", "count"),
)
print("Repeated vs Novel tokens:")
print(rep_stats.to_string(float_format="%.4f"))
print(
    f"\nRepetition benefit: {rep_stats.loc[False, 'mean_loss'] - rep_stats.loc[True, 'mean_loss']:.3f} nats"
)

# Break down by category
fig, ax = plt.subplots(figsize=(12, 5))
rep_by_cat = df.groupby(["category", "seen_in_context"])["loss"].mean().unstack()
rep_by_cat.plot(kind="bar", ax=ax, color=["coral", "steelblue"])
ax.set_ylabel("Mean Loss (nats)")
ax.set_title("Mean Loss: Novel vs Repeated Tokens by Category")
ax.legend(["Novel (not in context)", "Repeated (seen in context)"])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- In-context learning: can the model re-use novel names/terms? ---
# We construct 10 short passages where a rare/novel word (name, place, term)
# is introduced early, then must be predicted again later.
# For each passage we measure per-token loss and highlight the target word
# on its first vs subsequent appearances.

test_passages = [
    (
        "The village of Grothenvale was founded in 1342. Traders from the east often visited Grothenvale to buy rare spices.",
        "Grothenvale",
    ),
    (
        "Professor Yelinka Tarvo published her findings last spring. Many colleagues cited Tarvo's work in their own papers.",
        "Tarvo",
    ),
    (
        "The enzyme brixomaltase breaks down complex sugars in the gut. Patients with low brixomaltase levels often experience bloating.",
        "brixomaltase",
    ),
    (
        "A new satellite called Peristol-7 was launched on Tuesday. Peristol-7 will orbit Earth every ninety minutes.",
        "Peristol-7",
    ),
    (
        "In the kingdom of Zelaphon, magic was forbidden by law. The people of Zelaphon learned to hide their powers.",
        "Zelaphon",
    ),
    (
        "Dr. Fumihiro Kazetani won the prize for his work on quantum networks. Kazetani thanked his students in the acceptance speech.",
        "Kazetani",
    ),
    (
        "The Moldavite Corporation announced a merger with Synthelix. Analysts predicted that Moldavite would dominate the market.",
        "Moldavite",
    ),
    (
        "A rare flower called the crimson vellarose blooms only in December. Botanists travel far to study the vellarose.",
        "vellarose",
    ),
    (
        "The protocol known as QRST-4 was designed for encrypted mesh networks. Engineers praised QRST-4 for its simplicity.",
        "QRST-4",
    ),
    (
        "Chef Romilda Basquenet invented a new dessert using fermented plums. The dish, named after Basquenet, became an instant classic.",
        "Basquenet",
    ),
]

print(
    f"Testing {len(test_passages)} constructed passages for in-context word repetition\n"
)

results = []

for passage_idx, (passage, novel_word) in enumerate(test_passages):
    # Tokenize with BOS
    token_ids = [1] + sp_tok.encode(passage)  # BOS=1
    if len(token_ids) > SEQ_LEN:
        token_ids = token_ids[:SEQ_LEN]

    n_tokens = len(token_ids)
    padded = token_ids + [0] * (SEQ_LEN - n_tokens)
    x = torch.tensor(padded, device=device).unsqueeze(0)

    logits = get_logits(x)  # (SEQ_LEN, V)
    log_probs = F.log_softmax(logits.float(), dim=-1)

    per_token_loss = []
    for pos in range(1, n_tokens):
        target_id = token_ids[pos]
        lp = log_probs[pos - 1, target_id].item()
        per_token_loss.append(-lp)

    # --- Robust occurrence detection via decoded text matching ---
    # Decode each token to its piece string so we can find the novel word
    # by matching the concatenated text, regardless of tokenization splits.
    pieces = [sp_tok.decode([token_ids[i]]) for i in range(n_tokens)]

    # Build cumulative character offsets for each token
    decoded_full = "".join(pieces)
    char_offsets = []  # (char_start, char_end) for each token
    pos_c = 0
    for p in pieces:
        char_offsets.append((pos_c, pos_c + len(p)))
        pos_c += len(p)

    # Find all character-level occurrences of the novel word (case-sensitive)
    occurrences = []  # list of (token_start_idx, token_end_idx)
    search_start = 0
    while True:
        idx = decoded_full.find(novel_word, search_start)
        if idx == -1:
            break
        # Map character range [idx, idx+len(novel_word)) back to token indices
        word_char_end = idx + len(novel_word)
        tok_start = None
        tok_end = None
        for ti, (cs, ce) in enumerate(char_offsets):
            if cs <= idx < ce and tok_start is None:
                tok_start = ti
            if cs < word_char_end <= ce:
                tok_end = ti + 1
                break
        if tok_start is not None and tok_end is not None:
            occurrences.append((tok_start, tok_end))
        search_start = idx + 1

    # Collect loss for each occurrence
    occ_losses = []
    for occ_idx, (tok_start, tok_end) in enumerate(occurrences):
        # per_token_loss[i] = loss for predicting token_ids[i+1]
        # loss for token at position tok_start is per_token_loss[tok_start - 1]
        token_losses = [
            per_token_loss[ti - 1]
            for ti in range(tok_start, tok_end)
            if 0 <= ti - 1 < len(per_token_loss)
        ]
        mean_loss = np.mean(token_losses) if token_losses else float("nan")
        total_loss = np.sum(token_losses) if token_losses else float("nan")
        n_toks = tok_end - tok_start
        occ_losses.append(
            {
                "occurrence": occ_idx + 1,
                "tok_start": tok_start,
                "tok_end": tok_end,
                "mean_loss": mean_loss,
                "total_loss": total_loss,
                "n_tokens": n_toks,
            }
        )

    results.append(
        {
            "passage_idx": passage_idx,
            "novel_word": novel_word,
            "n_tokens": n_tokens,
            "mean_passage_loss": np.mean(per_token_loss),
            "occurrences": occ_losses,
            "all_token_ids": token_ids,
            "all_losses": per_token_loss,
            "char_offsets": char_offsets,
            "decoded_full": decoded_full,
        }
    )

    # Print summary
    print(f'[{passage_idx}] "{novel_word}" — {len(occ_losses)} occurrences found')
    for occ in occ_losses:
        tag = (
            "1st (intro)"
            if occ["occurrence"] == 1
            else f"{occ['occurrence']}nd+ (repeat)"
        )
        print(
            f"    {tag}: mean loss = {occ['mean_loss']:.3f} nats, total = {occ['total_loss']:.3f} nats ({occ['n_tokens']} tokens, pos {occ['tok_start']}-{occ['tok_end'] - 1})"
        )

# --- Aggregate: first vs repeat ---
first_losses = []
repeat_losses = []
for r in results:
    for occ in r["occurrences"]:
        if occ["occurrence"] == 1:
            first_losses.append(occ["mean_loss"])
        else:
            repeat_losses.append(occ["mean_loss"])

print(f"\n{'=' * 60}")
print(
    f"Aggregate over {len(results)} passages ({len(first_losses)} with >=1 occurrence, {len(repeat_losses)} with repeat):"
)
print(
    f"  First mention:  mean loss = {np.mean(first_losses):.3f} ± {np.std(first_losses):.3f} nats (n={len(first_losses)})"
)
print(
    f"  Repeat mention: mean loss = {np.mean(repeat_losses):.3f} ± {np.std(repeat_losses):.3f} nats (n={len(repeat_losses)})"
)
print(
    f"  Repetition benefit: {np.mean(first_losses) - np.mean(repeat_losses):.3f} nats"
)
if np.mean(first_losses) > 0:
    print(
        f"  Relative reduction: {(np.mean(first_losses) - np.mean(repeat_losses)) / np.mean(first_losses) * 100:.1f}%"
    )

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-passage comparison
ax = axes[0]
# Only plot passages with 2+ occurrences
plotable = [r for r in results if len(r["occurrences"]) >= 2]
x_pos = np.arange(len(plotable))
first_vals = [r["occurrences"][0]["mean_loss"] for r in plotable]
repeat_vals = [r["occurrences"][1]["mean_loss"] for r in plotable]
labels = [r["novel_word"] for r in plotable]

bar_w = 0.35
ax.bar(x_pos - bar_w / 2, first_vals, bar_w, label="1st mention", color="coral")
ax.bar(x_pos + bar_w / 2, repeat_vals, bar_w, label="2nd mention", color="steelblue")
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Mean Loss (nats)")
ax.set_title("Novel Word Loss: 1st vs 2nd Mention")
ax.legend()

# Right: token-level loss for one example, pick first with 2 occurrences
ax = axes[1]
ex = plotable[0]
ex_losses = ex["all_losses"]
ex_token_ids = ex["all_token_ids"]
ex_n = len(ex_token_ids)

# Highlight token positions belonging to any occurrence of the novel word
highlight_positions = set()
for occ in ex["occurrences"]:
    for ti in range(occ["tok_start"], occ["tok_end"]):
        if 0 <= ti - 1 < len(ex_losses):
            highlight_positions.add(ti - 1)  # index into ex_losses

colors = [
    "coral" if i in highlight_positions else "lightgray" for i in range(len(ex_losses))
]
ax.bar(range(len(ex_losses)), ex_losses, color=colors, width=1.0, edgecolor="none")
ax.set_xlabel("Token position")
ax.set_ylabel("Loss (nats)")
ax.set_title(f'Token-level loss: "{ex["novel_word"]}" passage')
from matplotlib.patches import Patch

ax.legend(
    handles=[
        Patch(color="coral", label=f'"{ex["novel_word"]}" tokens'),
        Patch(color="lightgray", label="Other tokens"),
    ]
)

plt.tight_layout()
plt.savefig(PLOT_DIR / "incontext_novel_word_repetition.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: incontext_novel_word_repetition.png")


In [ ]:
# --- Sample sentence completions: spelling, grammar, semantics ---
# Feed the model partial sentences and let it complete them autoregressively.
# We categorize prompts to probe different failure modes.


@torch.no_grad()
def sample_completion(prompt_text, max_new_tokens=60, temperature=1.0, top_k=50):
    """Autoregressively sample a completion for the given text prompt."""
    prompt_ids = [1] + sp_tok.encode(prompt_text)  # BOS + prompt
    generated = list(prompt_ids)

    for _ in range(max_new_tokens):
        if len(generated) >= SEQ_LEN:
            break
        # Pad to SEQ_LEN, run forward pass
        padded = generated + [0] * (SEQ_LEN - len(generated))
        x = torch.tensor(padded, device=device).unsqueeze(0)
        logits = get_logits(x)  # (SEQ_LEN, V)
        next_logits = logits[len(generated) - 1]  # logits predicting next token

        # Temperature + top-k sampling
        next_logits = next_logits / temperature
        if top_k > 0:
            topk_vals, topk_idx = torch.topk(next_logits, top_k)
            mask = torch.full_like(next_logits, float("-inf"))
            mask.scatter_(0, topk_idx, topk_vals)
            next_logits = mask

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()

        # Stop on BOS (document boundary) or pad
        if next_token <= 1:
            break
        generated.append(next_token)

    completion_ids = generated[len(prompt_ids) :]
    return sp_tok.decode(completion_ids)


# --- Prompt categories ---
prompts = {
    "Spelling / rare words": [
        "The archaeologist discovered an ancient",
        "She carefully arranged the chrysanthemums",
        "The pharmaceutical company developed a new",
        "His Mediterranean vacation included visits to",
        "The bureaucratic process for obtaining a",
    ],
    "Grammar (agreement, tense, structure)": [
        "Neither the teacher nor the students",
        "If she had known about the problem earlier,",
        "The committee, along with its advisors,",
        "Each of the participants was asked to",
        "Not only did he finish the project, but",
    ],
    "Semantic coherence": [
        "The sun set over the ocean as the fisherman",
        "After years of research, the scientist finally",
        "The old library on the corner had been closed for",
        "When the power went out during the storm,",
        "The dog wagged its tail because",
    ],
    "Factual / world knowledge": [
        "The capital of France is known for its",
        "Water freezes at a temperature of",
        "The theory of evolution was first proposed by",
        "The largest planet in our solar system is",
        "Photosynthesis is the process by which plants",
    ],
}

# --- Generate completions ---
N_SAMPLES = 3  # samples per prompt for diversity
print("Sampling completions (temperature=0.8, top_k=50)...\n")

all_results = {}
for category, category_prompts in prompts.items():
    print(f"{'=' * 70}")
    print(f"  {category}")
    print(f"{'=' * 70}")
    cat_results = []
    for prompt in category_prompts:
        samples = []
        for s in range(N_SAMPLES):
            completion = sample_completion(
                prompt, max_new_tokens=40, temperature=0.8, top_k=50
            )
            samples.append(completion)
        cat_results.append({"prompt": prompt, "samples": samples})

        print(f'\n  Prompt: "{prompt}..."')
        for s_idx, comp in enumerate(samples):
            # Truncate at first newline or sentence end for readability
            display = comp.strip()
            if len(display) > 200:
                display = display[:200] + "..."
            print(f"    [{s_idx + 1}] {display}")

    all_results[category] = cat_results

# --- Also do greedy decoding (temperature -> 0) for deterministic output ---
print(f"\n{'=' * 70}")
print(f"  Greedy decoding (deterministic, one sample each)")
print(f"{'=' * 70}")
greedy_results = []
for category, category_prompts in prompts.items():
    for prompt in category_prompts:
        completion = sample_completion(
            prompt, max_new_tokens=40, temperature=0.01, top_k=1
        )
        greedy_results.append(
            {"prompt": prompt, "category": category, "completion": completion}
        )

        display = completion.strip()[:200]
        print(f'\n  [{category[:8]}] "{prompt}..."')
        print(f"    → {display}")

print(
    f"\n\nTotal: {sum(len(v) for v in prompts.values())} prompts × {N_SAMPLES} samples + greedy = {sum(len(v) for v in prompts.values()) * (N_SAMPLES + 1)} completions"
)


In [ ]:
# --- Document warm-up trajectories ---
# Overlay per-document loss curves for the first 50 tokens
fig, ax = plt.subplots(figsize=(14, 5))

# Sample some documents to plot individually
doc_ids_sample = doc_stats_valid.sample(
    min(100, len(doc_stats_valid)), random_state=42
).index
for doc_id in doc_ids_sample:
    doc_df = df[(df["doc_id"] == doc_id) & (df["doc_position"] <= 50)]
    if len(doc_df) >= 10:
        ax.plot(
            doc_df["doc_position"],
            doc_df["loss"],
            alpha=0.05,
            color="steelblue",
            linewidth=0.5,
        )

# Overlay the mean
mean_warmup = df[df["doc_position"] <= 50].groupby("doc_position")["loss"].mean()
ax.plot(mean_warmup.index, mean_warmup.values, color="red", linewidth=2.5, label="Mean")
ax.set_xlabel("Position in Document")
ax.set_ylabel("Loss (nats)")
ax.set_title("Document Warm-up Trajectories (first 50 tokens)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Surprise Analysis

What are the highest individual losses? What patterns do they share?

In [ ]:
# --- Top 50 highest individual losses with context ---
print("TOP 50 HIGHEST INDIVIDUAL LOSSES")
print("=" * 100)
top_surprises = df.nlargest(50, "loss")
for rank, (idx, row) in enumerate(top_surprises.iterrows()):
    seq = int(row["seq_idx"])
    pos = int(row["position"])
    x, y = get_val_batch(seq)

    # Context: 20 preceding tokens
    start_ctx = max(0, pos - 20)
    ctx_tokens = x[0, start_ctx : pos + 1].tolist()
    ctx_text = decode_tokens(ctx_tokens)

    target_piece = id_to_piece[int(row["token_id"])]
    pred_piece = id_to_piece[int(row["top1_pred"])]

    # Get top-3 predictions
    logits = get_logits(x[0:1, :])
    top3_ids = logits[pos].topk(3).indices.tolist()
    top3_pieces = [id_to_piece[t] for t in top3_ids]

    print(
        f"#{rank + 1:2d} loss={row['loss']:.2f} ent={row['entropy']:.2f} cat={row['category']} doc_pos={row['doc_position']}"
    )
    print(f"     ...{ctx_text[-50:]}>>>>{target_piece!r}<<<<")
    print(f"     top3: {top3_pieces}")
    print()

In [ ]:
# --- Automatic clustering of high-loss tokens ---
top1000 = df.nlargest(1000, "loss").copy()
top1000["doc_start"] = top1000["doc_position"] < 5

# Cross-tabulation: category x seen_in_context x doc_start
ct = (
    top1000.groupby(["category", "seen_in_context", "doc_start"])
    .agg(
        count=("loss", "count"),
        mean_loss=("loss", "mean"),
        total_loss=("loss", "sum"),
    )
    .reset_index()
)
ct = ct.sort_values("total_loss", ascending=False)
print(
    "Top 1000 highest-loss tokens — grouped by (category, seen_in_context, doc_start):"
)
print(ct.head(15).to_string(index=False))

# Heatmap: category x doc_position_bin
top1000["dp_bin"] = pd.cut(
    top1000["doc_position"],
    bins=[0, 2, 5, 20, 50, 200, 10000],
    labels=["0-2", "3-5", "6-20", "21-50", "51-200", "200+"],
)
heatmap_data = top1000.pivot_table(
    values="loss", index="category", columns="dp_bin", aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_xlabel("Document Position Bin")
ax.set_ylabel("Token Category")
ax.set_title("Mean Loss of Top-1000 Surprises by Category and Position")
plt.colorbar(im, label="Mean Loss (nats)")
# Annotate cells
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        if not np.isnan(val):
            ax.text(
                j,
                i,
                f"{val:.1f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white"
                if val > heatmap_data.values[~np.isnan(heatmap_data.values)].mean()
                else "black",
            )
plt.tight_layout()
plt.show()

In [ ]:
# --- Recovery after high-loss events: misspelling vs word-start vs other ---
# Classify high-loss tokens by error type and compare how quickly the model recovers.
# "Misspelling-like": word_internal token with high loss where model predicted a different continuation
# "Word-start surprise": the model was surprised by which word came next
# "Other": punctuation, BOS, byte_fallback surprises

WINDOW = 30  # tokens to track after surprise
threshold = df["loss"].quantile(0.95)

# Classify each high-loss event
misspelling_idx = []
word_start_idx = []
other_surprise_idx = []

for i, row in df[df["loss"] > threshold].iterrows():
    pos = int(row["position"])
    if pos + WINDOW >= SEQ_LEN:
        continue
    target_cat = row["category"]
    pred_id = int(row["top1_pred"])
    pred_cat = token_meta.loc[pred_id, "category"]

    if target_cat == "word_internal" and pred_cat in (
        "word_internal",
        "space_word",
        "punctuation",
    ):
        misspelling_idx.append(i)
    elif target_cat in ("space_word", "uppercase_start"):
        word_start_idx.append(i)
    else:
        other_surprise_idx.append(i)

# Control: typical-loss tokens
typical = df[
    (df["loss"] > df["loss"].median() * 0.8) & (df["loss"] < df["loss"].median() * 1.2)
]
typical = typical[typical["position"] + WINDOW < SEQ_LEN]
control_idx = typical.sample(min(50000, len(typical)), random_state=42).index.tolist()


def build_recovery_trajectories(indices):
    """For each index, collect the loss at that position and the next WINDOW positions."""
    trajs = []
    for i in indices:
        row = df.loc[i]
        seq = int(row["seq_idx"])
        pos = int(row["position"])
        # Flat index into the all_losses array
        flat_start = seq * SEQ_LEN + pos
        flat_end = flat_start + WINDOW + 1
        trajs.append(all_losses[flat_start:flat_end])
    return np.array(trajs) if trajs else np.empty((0, WINDOW + 1))


traj_misspelling = build_recovery_trajectories(misspelling_idx)
traj_word_start = build_recovery_trajectories(word_start_idx)
traj_other = build_recovery_trajectories(other_surprise_idx)
traj_control = build_recovery_trajectories(control_idx)

print(f"Recovery trajectories (p95 threshold = {threshold:.2f}):")
print(f"  Misspelling-like (word_internal): {len(traj_misspelling)}")
print(f"  Word-start surprise:              {len(traj_word_start)}")
print(f"  Other surprise:                   {len(traj_other)}")
print(f"  Control (typical loss):           {len(traj_control)}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
offsets = np.arange(WINDOW + 1)
global_mean = df["loss"].mean()

curves = [
    (traj_misspelling, f"Misspelling-like (n={len(traj_misspelling)})", "red"),
    (traj_word_start, f"Word-start surprise (n={len(traj_word_start)})", "orange"),
    (traj_other, f"Other surprise (n={len(traj_other)})", "purple"),
    (traj_control, f"Control/typical (n={len(traj_control)})", "gray"),
]

# Left: mean
for trajs, label, color in curves:
    if len(trajs) == 0:
        continue
    axes[0].plot(offsets, trajs.mean(axis=0), label=label, color=color, linewidth=2)
axes[0].axhline(
    global_mean,
    color="black",
    linestyle=":",
    alpha=0.4,
    label=f"Global mean ({global_mean:.2f})",
)
axes[0].set_xlabel("Tokens after surprise (0 = surprise token)")
axes[0].set_ylabel("Mean Loss (nats)")
axes[0].set_title("Recovery After High-Loss Events (Mean)")
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, WINDOW)

# Right: median + IQR
for trajs, label, color in curves:
    if len(trajs) == 0:
        continue
    med = np.median(trajs, axis=0)
    p25 = np.percentile(trajs, 25, axis=0)
    p75 = np.percentile(trajs, 75, axis=0)
    axes[1].plot(offsets, med, label=label, color=color, linewidth=2)
    axes[1].fill_between(offsets, p25, p75, color=color, alpha=0.1)
axes[1].axhline(
    global_mean, color="black", linestyle=":", alpha=0.4, label="Global mean"
)
axes[1].set_xlabel("Tokens after surprise (0 = surprise token)")
axes[1].set_ylabel("Median Loss (nats)")
axes[1].set_title("Recovery After High-Loss Events (Median + IQR)")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, WINDOW)

plt.tight_layout()
plt.show()

# Numeric table
c_mean = traj_control.mean(axis=0)
print(
    f"\n{'Type':<28} {'t=0':>7} {'t=1':>7} {'t=2':>7} {'t=3':>7} {'t=5':>7} {'t=10':>7} {'t=20':>7}"
)
for trajs, label in [
    (traj_misspelling, "Misspelling-like"),
    (traj_word_start, "Word-start surprise"),
    (traj_other, "Other surprise"),
    (traj_control, "Control"),
]:
    if len(trajs) == 0:
        continue
    m = trajs.mean(axis=0)
    print(
        f"{label:<28} {m[0]:>7.3f} {m[1]:>7.3f} {m[2]:>7.3f} {m[3]:>7.3f} {m[5]:>7.3f} {m[10]:>7.3f} {m[20]:>7.3f}"
    )

print(f"\nDifference from control (mean):")
for trajs, label in [
    (traj_misspelling, "Misspelling-like"),
    (traj_word_start, "Word-start surprise"),
    (traj_other, "Other surprise"),
]:
    if len(trajs) == 0:
        continue
    diff = trajs.mean(axis=0) - c_mean
    print(
        f"{label:<28} {diff[0]:>+7.3f} {diff[1]:>+7.3f} {diff[2]:>+7.3f} {diff[3]:>+7.3f} {diff[5]:>+7.3f} {diff[10]:>+7.3f} {diff[20]:>+7.3f}"
    )

# Show sample misspelling-like errors with word context
print(f"\n--- Sample misspelling-like errors (word prefix + actual vs predicted) ---")
count = 0
for i in misspelling_idx[:40]:
    row = df.loc[i]
    seq = int(row["seq_idx"])
    pos = int(row["position"])
    x, y = get_val_batch(seq)
    # Walk backwards to find word start
    word_pieces = []
    for p in range(pos, max(pos - 20, -1), -1):
        tid = int(x[0, p])
        piece = id_to_piece[tid]
        word_pieces.insert(0, piece)
        if piece.startswith("\u2581") or tid == 1:
            break
    prefix = "".join(word_pieces).replace("\u2581", "")
    actual = id_to_piece[int(row["token_id"])].replace("\u2581", " ")
    predicted = id_to_piece[int(row["top1_pred"])].replace("\u2581", " ")
    loss = row["loss"]
    print(
        f"  loss={loss:.1f} | '{prefix}' + actual='{actual}' vs predicted='{predicted}'"
    )
    count += 1
    if count >= 20:
        break


In [ ]:
# --- Causal BPB savings from correcting misspellings ---
# Estimate: if we detected and corrected spelling mistakes causally (only using
# information available up to the current position), how much BPB could we recover
# on subsequent tokens?
#
# Approach:
# 1. Extract complete words from validation sequences
# 2. Spellcheck them (pyspellchecker, with strict filtering to reduce false positives)
# 3. For each misspelling: build a "corrected" input sequence by replacing the
#    misspelled word tokens with the corrected word's tokenization
# 4. Re-run the model and compare losses on SUFFIX tokens only (causal constraint:
#    we can only benefit after the word is complete and correction is applied)
# 5. The corrected word may tokenize into a different number of tokens, so we
#    carefully align the suffix targets to ensure we compare identical target tokens.
#
# NOTE: pyspellchecker is a simple frequency dictionary. It does NOT know proper nouns,
# so it will flag names like "Kunalic" -> "Lunatic". We mitigate this with strict filters
# but some false positives remain. Results should be interpreted as an upper/lower bound.

from spellchecker import SpellChecker

spell = SpellChecker()
SUFFIX_WINDOW = 30  # tokens after the word to compare


def edit_distance(s1, s2):
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]


@torch.no_grad()
def per_token_nll(x_input, y_target):
    logits = get_logits(x_input)
    log_probs = F.log_softmax(logits.float(), dim=-1)
    return -log_probs[torch.arange(logits.shape[0]), y_target[0]].cpu().numpy()


# --- Step 1: Find misspellings with strict filtering ---
print("Step 1: Scanning for misspellings (strict filtering)...")
misspellings = []
total_words = 0

for batch_idx in tqdm(range(N_BATCHES), desc="Scanning words"):
    x, y = get_val_batch(batch_idx)
    input_ids = x[0].tolist()

    # Segment into words at leading-space boundaries
    word_start = 0
    words_in_seq = []
    for pos in range(SEQ_LEN):
        piece = id_to_piece[input_ids[pos]]
        if (piece.startswith("\u2581") or input_ids[pos] == 1) and pos > word_start:
            words_in_seq.append((word_start, pos, input_ids[word_start:pos]))
            word_start = pos
    if word_start < SEQ_LEN:
        words_in_seq.append((word_start, SEQ_LEN, input_ids[word_start:SEQ_LEN]))

    for ws, we, tok_ids in words_in_seq:
        decoded = sp_tok.decode(tok_ids).strip()
        # Strict filters to avoid false positives:
        if not decoded.isalpha():
            continue  # no punctuation, numbers, hyphens
        if len(decoded) < 4 or len(decoded) > 25:
            continue  # skip very short/long
        if decoded.isupper() and len(decoded) <= 5:
            continue  # skip abbreviations
        if decoded[0].isupper() and len(decoded) < 6:
            continue  # skip short capitalized (likely names)
        total_words += 1

        lower = decoded.lower()
        if lower not in spell.unknown([lower]):
            continue  # word is known, skip
        correction = spell.correction(lower)
        if correction is None or correction == lower:
            continue
        ed = edit_distance(lower, correction)
        if ed > 2:
            continue  # only small edits

        # Preserve casing
        if decoded[0].isupper() and decoded[1:].islower():
            correction = correction.capitalize()
        elif decoded.isupper():
            correction = correction.upper()

        misspellings.append(
            {
                "batch_idx": batch_idx,
                "word_start": ws,
                "word_end": we,
                "misspelled": decoded,
                "corrected": correction,
                "token_ids": tok_ids,
                "edit_distance": ed,
            }
        )

print(f"Words checked: {total_words:,}")
print(
    f"Misspellings found: {len(misspellings)} ({len(misspellings) / max(total_words, 1):.2%})"
)

# --- Step 2: Causal re-scoring ---
print(
    f"\nStep 2: Re-scoring {len(misspellings)} corrections (suffix window = {SUFFIX_WINDOW})..."
)

results = []
n_skipped = 0

for m in tqdm(misspellings, desc="Re-scoring"):
    batch_idx = m["batch_idx"]
    ws, we = m["word_start"], m["word_end"]

    if we + SUFFIX_WINDOW > SEQ_LEN:
        n_skipped += 1
        continue

    x_orig, y_orig = get_val_batch(batch_idx)
    global_start = batch_idx * SEQ_LEN
    full_orig = val_tokens[global_start : global_start + SEQ_LEN + 1].tolist()

    # Tokenize corrected word (preserve leading space if original had one)
    first_piece = id_to_piece[full_orig[ws]]
    has_space = first_piece.startswith("\u2581")
    corrected_text = (" " + m["corrected"]) if has_space else m["corrected"]
    corrected_tok_ids = sp_tok.encode(corrected_text)
    n_corr = len(corrected_tok_ids)

    # Build corrected full sequence
    full_corrected = full_orig[:ws] + corrected_tok_ids + full_orig[we:]
    if len(full_corrected) < SEQ_LEN + 1:
        n_skipped += 1
        continue

    x_corr = torch.tensor(full_corrected[:SEQ_LEN], dtype=torch.long).unsqueeze(0)
    y_corr = torch.tensor(full_corrected[1 : SEQ_LEN + 1], dtype=torch.long).unsqueeze(
        0
    )

    # Compute losses
    orig_losses = per_token_nll(x_orig, y_orig)
    corr_losses = per_token_nll(x_corr, y_corr)

    # Suffix comparison: aligned target tokens
    orig_suffix_start = we
    corr_suffix_start = ws + n_corr
    n_suffix = min(
        SUFFIX_WINDOW, SEQ_LEN - we, len(full_corrected) - 1 - corr_suffix_start
    )
    if n_suffix <= 0:
        n_skipped += 1
        continue

    # Verify target alignment
    orig_targets = y_orig[0, orig_suffix_start : orig_suffix_start + n_suffix].tolist()
    corr_targets = y_corr[0, corr_suffix_start : corr_suffix_start + n_suffix].tolist()
    if orig_targets != corr_targets:
        n_skipped += 1
        continue

    orig_suffix = orig_losses[orig_suffix_start : orig_suffix_start + n_suffix]
    corr_suffix = corr_losses[corr_suffix_start : corr_suffix_start + n_suffix]

    results.append(
        {
            "misspelled": m["misspelled"],
            "corrected": m["corrected"],
            "edit_distance": m["edit_distance"],
            "orig_loss": float(orig_suffix.sum()),
            "corr_loss": float(corr_suffix.sum()),
            "savings_nats": float(orig_suffix.sum() - corr_suffix.sum()),
            "n_suffix": n_suffix,
            "orig_per_offset": orig_suffix.tolist(),
            "corr_per_offset": corr_suffix.tolist(),
        }
    )

print(f"Processed: {len(results)}, Skipped: {n_skipped}")

# --- Step 3: Analysis ---
savings_arr = np.array([r["savings_nats"] for r in results])
total_suffix_tokens = sum(r["n_suffix"] for r in results)
total_orig_loss = sum(r["orig_loss"] for r in results)
total_corr_loss = sum(r["corr_loss"] for r in results)

total_scanned_tokens = N_BATCHES * SEQ_LEN
# Use actual byte count from the DataFrame for BPB scaling
total_scanned_bytes = df["bytes"].sum()
bpb_savings = (total_orig_loss - total_corr_loss) / ln2 / total_scanned_bytes

print(f"\n{'=' * 70}")
print(f"CAUSAL SPELLCHECK SAVINGS")
print(f"{'=' * 70}")
print(f"Misspellings corrected: {len(results)}")
print(f"Suffix tokens compared: {total_suffix_tokens:,}")
print(f"Mean suffix loss (original):  {total_orig_loss / total_suffix_tokens:.4f} nats")
print(f"Mean suffix loss (corrected): {total_corr_loss / total_suffix_tokens:.4f} nats")
print(
    f"Total savings: {total_orig_loss - total_corr_loss:.1f} nats = {(total_orig_loss - total_corr_loss) / ln2:.1f} bits"
)
print(f"Estimated BPB impact: {bpb_savings:+.6f}")
print(f"\nPer-correction distribution:")
print(f"  Mean:   {savings_arr.mean():+.3f} nats")
print(f"  Median: {np.median(savings_arr):+.3f} nats")
print(f"  Helps (>0): {(savings_arr > 0).sum()} ({(savings_arr > 0).mean():.1%})")
print(f"  Hurts (<0): {(savings_arr < 0).sum()} ({(savings_arr < 0).mean():.1%})")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram of per-correction savings
axes[0].hist(savings_arr, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].axvline(
    savings_arr.mean(),
    color="green",
    linewidth=2,
    label=f"Mean = {savings_arr.mean():+.2f} nats",
)
axes[0].set_xlabel("Savings (nats) per correction")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Per-Correction Savings Distribution (n={len(results)})")
axes[0].legend()

# Recovery curves: mean loss at each suffix offset, original vs corrected
max_offsets = min(SUFFIX_WINDOW, 30)
orig_by_offset = [[] for _ in range(max_offsets)]
corr_by_offset = [[] for _ in range(max_offsets)]
for r in results:
    for k in range(min(len(r["orig_per_offset"]), max_offsets)):
        orig_by_offset[k].append(r["orig_per_offset"][k])
        corr_by_offset[k].append(r["corr_per_offset"][k])

offsets = list(range(max_offsets))
orig_means = [np.mean(orig_by_offset[k]) for k in offsets]
corr_means = [np.mean(corr_by_offset[k]) for k in offsets]
axes[1].plot(offsets, orig_means, "r-o", markersize=3, label="Original (misspelled)")
axes[1].plot(offsets, corr_means, "b-o", markersize=3, label="Corrected")
axes[1].fill_between(
    offsets,
    corr_means,
    orig_means,
    alpha=0.15,
    color="green" if np.mean(orig_means) > np.mean(corr_means) else "red",
)
axes[1].axhline(
    df["loss"].mean(), color="gray", linestyle=":", alpha=0.5, label="Global mean"
)
axes[1].set_xlabel("Tokens after word end")
axes[1].set_ylabel("Mean Loss (nats)")
axes[1].set_title("Suffix Loss: Original vs Corrected Prefix")
axes[1].legend(fontsize=8)

# Top helps vs top hurts
sorted_r = sorted(results, key=lambda r: r["savings_nats"], reverse=True)
top_helps = sorted_r[:10]
top_hurts = sorted_r[-10:]

labels_h = [f"'{r['misspelled']}'->\n'{r['corrected']}'" for r in top_helps]
labels_u = [f"'{r['misspelled']}'->\n'{r['corrected']}'" for r in top_hurts]
all_labels = labels_h + [""] + labels_u
all_vals = (
    [r["savings_nats"] for r in top_helps]
    + [0]
    + [r["savings_nats"] for r in top_hurts]
)
colors = ["green" if v > 0 else "red" if v < 0 else "white" for v in all_vals]
axes[2].barh(
    range(len(all_vals)), all_vals, color=colors, edgecolor="black", linewidth=0.5
)
axes[2].set_yticks(range(len(all_vals)))
axes[2].set_yticklabels(all_labels, fontsize=6)
axes[2].axvline(0, color="black", linewidth=0.5)
axes[2].set_xlabel("Savings (nats)")
axes[2].set_title("Best Corrections vs Worst False Positives")
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(PLOT_DIR / "causal_spellcheck_savings.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"\nConclusion: Naive causal spellcheck correction yields {bpb_savings:+.6f} BPB."
)
print(
    f"The spellchecker (pyspellchecker) has high false positive rate on proper nouns/names,"
)
print(f"which damages more BPB than genuine corrections recover.")
print(
    f"A model-aware spellchecker (using the LM's own predictions to detect errors) would"
)
print(f"likely perform better by avoiding false positives on in-distribution text.")

In [ ]:
# --- Model-as-oracle misspelling detection and causal BPB savings ---
# Instead of an external spellchecker (which flags proper nouns as errors),
# use the model's OWN predictions to detect misspellings:
#
# 1. For each word in the validation data, check if it's a known English word
# 2. If it's unknown, greedily reconstruct what the model "wanted" to write
#    by taking top-1 predictions at each word-internal position
# 3. If the model's version IS a known word -> high-confidence misspelling
# 4. Causally re-score: replace misspelled word with model's predicted word
#
# This avoids false positives on proper nouns because the model already learned
# them from training data — it won't try to "correct" names it knows.

from spellchecker import SpellChecker

spell_oracle = SpellChecker()

# We need per-position top-1 predictions. Reuse the already-collected data from df.
# df has: token_id (actual), top1_pred (model's prediction), position, seq_idx

# === Step 1: Reconstruct model's preferred words at unknown-word positions ===
print("Step 1: Model-as-oracle misspelling detection...")

model_detections = []
n_unknown_words = 0
n_known_words = 0

for batch_idx in tqdm(range(N_BATCHES), desc="Model-oracle scan"):
    x, y = get_val_batch(batch_idx)
    logits_all = get_logits(x)  # (SEQ_LEN, V)
    top1_preds = logits_all.argmax(dim=-1).cpu().numpy()  # model's top-1 at each pos

    input_ids = x[0].tolist()
    target_ids = y[0].tolist()

    # Segment into words
    word_boundaries = []  # (start, end) in input sequence
    word_start = 0
    for pos in range(SEQ_LEN):
        piece = id_to_piece[input_ids[pos]]
        if (piece.startswith("\u2581") or input_ids[pos] == 1) and pos > word_start:
            word_boundaries.append((word_start, pos))
            word_start = pos
    if word_start < SEQ_LEN:
        word_boundaries.append((word_start, SEQ_LEN))

    for ws, we in word_boundaries:
        # Decode the actual word
        actual_word = sp_tok.decode(input_ids[ws:we]).strip()
        if not actual_word.isalpha() or len(actual_word) < 4 or len(actual_word) > 25:
            continue
        if actual_word.isupper() and len(actual_word) <= 5:
            continue
        # Skip capitalized words — likely proper nouns, not misspellings
        if actual_word[0].isupper():
            continue

        actual_lower = actual_word.lower()
        is_known = actual_lower not in spell_oracle.unknown([actual_lower])
        if is_known:
            n_known_words += 1
            continue
        n_unknown_words += 1

        # The word is unknown. Reconstruct what the model wanted to write.
        # At each target position within this word (ws to we-1), the model
        # predicted top1_preds[pos] instead of target_ids[pos].
        # Target at position pos corresponds to predicting input[pos+1],
        # i.e., target_ids[pos] = input_ids[pos+1] for pos < SEQ_LEN-1.
        #
        # The model's preferred word: take the prefix up to the word start,
        # then greedily use model predictions at each position.
        # But the model's prediction at pos depends on input[0..pos],
        # so after we substitute one token, subsequent predictions change.
        # For simplicity, we just check: does the model's single-step
        # top-1 at the FIRST divergence point lead to a known word?

        # Find first position within the word where model disagrees
        first_diverge = None
        for pos in range(ws, min(we, SEQ_LEN)):
            # target_ids[pos] is the next token (input_ids[pos+1] if pos+1 < SEQ_LEN)
            if pos < SEQ_LEN and target_ids[pos] != top1_preds[pos]:
                # Does this position correspond to a word-internal token?
                if pos >= ws:
                    first_diverge = pos
                    break

        if first_diverge is None:
            continue

        # Build model's version: take actual tokens up to divergence,
        # then model's prediction at divergence, then actual tokens after
        # (This is a one-token substitution — conservative but clean)
        model_word_tokens = list(input_ids[ws : first_diverge + 1]) + [
            int(top1_preds[first_diverge])
        ]
        # Add remaining actual tokens to complete the word
        model_word_tokens += input_ids[first_diverge + 2 : we]

        # But wait: the model's predicted token might end the word (have leading space)
        # or continue it differently. Let's just decode and check.
        model_word = sp_tok.decode(model_word_tokens).strip()

        # Check if model's version is a known word
        if not model_word.isalpha() or len(model_word) < 3:
            continue
        model_lower = model_word.lower()
        model_is_known = model_lower not in spell_oracle.unknown([model_lower])

        if model_is_known and not is_known:
            model_detections.append(
                {
                    "batch_idx": batch_idx,
                    "word_start": ws,
                    "word_end": we,
                    "actual_word": actual_word,
                    "model_word": model_word,
                    "diverge_pos": first_diverge,
                    "actual_token": id_to_piece[target_ids[first_diverge]],
                    "model_token": id_to_piece[top1_preds[first_diverge]],
                }
            )

print(f"Known words: {n_known_words:,}, Unknown words: {n_unknown_words:,}")
print(
    f"Model-detected misspellings (unknown actual, known model): {len(model_detections)}"
)
print(f"\nSamples:")
for d in model_detections[:20]:
    print(
        f"  '{d['actual_word']}' -> model wanted '{d['model_word']}' "
        f"(diverged at '{d['actual_token']}' vs '{d['model_token']}')"
    )

# === Step 2: Causal re-scoring for model-detected misspellings ===
print(f"\nStep 2: Causal re-scoring ({len(model_detections)} detections)...")
SUFFIX_WINDOW_ORACLE = 30

oracle_results = []
for d in tqdm(model_detections, desc="Oracle re-scoring"):
    batch_idx, ws, we = d["batch_idx"], d["word_start"], d["word_end"]
    if we + SUFFIX_WINDOW_ORACLE > SEQ_LEN:
        continue

    x_orig, y_orig = get_val_batch(batch_idx)
    global_start = batch_idx * SEQ_LEN
    full_orig = val_tokens[global_start : global_start + SEQ_LEN + 1].tolist()

    # Tokenize the model's preferred word
    first_piece = id_to_piece[full_orig[ws]]
    has_space = first_piece.startswith("\u2581")
    model_text = (" " + d["model_word"]) if has_space else d["model_word"]
    model_tok_ids = sp_tok.encode(model_text)
    n_corr = len(model_tok_ids)

    full_corrected = full_orig[:ws] + model_tok_ids + full_orig[we:]
    if len(full_corrected) < SEQ_LEN + 1:
        continue

    x_corr = torch.tensor(full_corrected[:SEQ_LEN], dtype=torch.long).unsqueeze(0)
    y_corr = torch.tensor(full_corrected[1 : SEQ_LEN + 1], dtype=torch.long).unsqueeze(
        0
    )

    orig_losses = per_token_nll(x_orig, y_orig)
    corr_losses = per_token_nll(x_corr, y_corr)

    corr_suffix_start = ws + n_corr
    n_suffix = min(
        SUFFIX_WINDOW_ORACLE, SEQ_LEN - we, len(full_corrected) - 1 - corr_suffix_start
    )
    if n_suffix <= 0:
        continue

    orig_targets = y_orig[0, we : we + n_suffix].tolist()
    corr_targets = y_corr[0, corr_suffix_start : corr_suffix_start + n_suffix].tolist()
    if orig_targets != corr_targets:
        continue

    orig_suffix = orig_losses[we : we + n_suffix]
    corr_suffix = corr_losses[corr_suffix_start : corr_suffix_start + n_suffix]
    savings = float(orig_suffix.sum() - corr_suffix.sum())

    oracle_results.append(
        {
            **d,
            "savings_nats": savings,
            "n_suffix": n_suffix,
            "n_orig_tokens": we - ws,
            "n_corr_tokens": n_corr,
        }
    )

# === Results ===
oracle_savings = np.array([r["savings_nats"] for r in oracle_results])
oracle_total_bits = oracle_savings.sum() / ln2
oracle_bpb = oracle_total_bits / df["bytes"].sum()

oracle_helps = oracle_savings > 0
oracle_helps_bits = oracle_savings[oracle_helps].sum() / ln2
oracle_helps_bpb = oracle_helps_bits / df["bytes"].sum()

print(f"\n{'=' * 70}")
print(f"MODEL-AS-ORACLE RESULTS")
print(f"{'=' * 70}")
print(f"Detections: {len(oracle_results)}")
print(f"Corrections that help: {oracle_helps.sum()} ({oracle_helps.mean():.1%})")
print(f"Corrections that hurt: {(~oracle_helps).sum()} ({(~oracle_helps).mean():.1%})")
print(f"Mean savings per detection: {oracle_savings.mean():.3f} nats")
print(f"\nNet BPB impact (all):    {oracle_bpb:+.6f}")
print(f"Oracle BPB (only helps): {oracle_helps_bpb:+.6f}")
print(f"Overall BPB:             {overall_bpb:.4f}")
print(
    f"Oracle fraction:         {abs(oracle_helps_bpb / overall_bpb):.3%} of total BPB"
)

# --- Plot comparison: pyspellchecker vs model-oracle ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: savings distributions side by side
try:
    spell_savings = np.array([e["savings_nats"] for e in per_example_results])
    axes[0].hist(
        spell_savings,
        bins=40,
        alpha=0.5,
        color="salmon",
        label="pyspellchecker",
        density=True,
    )
except NameError:
    spell_savings = np.array([])
axes[0].hist(
    oracle_savings,
    bins=40,
    alpha=0.5,
    color="steelblue",
    label="Model-as-oracle",
    density=True,
)
axes[0].axvline(0, color="black", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Savings (nats) per correction")
axes[0].set_ylabel("Density")
axes[0].set_title("Savings Distribution: External vs Model-Oracle")
axes[0].legend()

# Right: summary comparison
methods = [
    "pyspellchecker\n(naive)",
    "pyspellchecker\n(oracle-filtered)",
    "Model-as-oracle\n(all)",
    "Model-as-oracle\n(oracle-filtered)",
]
try:
    spell_all_bpb = (
        sum(e["savings_nats"] for e in per_example_results) / ln2 / df["bytes"].sum()
    )
    spell_oracle_bpb = (
        sum(e["savings_nats"] for e in per_example_results if e["savings_nats"] > 0)
        / ln2
        / df["bytes"].sum()
    )
except NameError:
    spell_all_bpb = 0
    spell_oracle_bpb = 0
values = [spell_all_bpb, spell_oracle_bpb, oracle_bpb, oracle_helps_bpb]
colors_bar = ["salmon", "lightsalmon", "steelblue", "lightblue"]
axes[1].bar(
    range(len(methods)), values, color=colors_bar, edgecolor="black", linewidth=0.5
)
axes[1].set_xticks(range(len(methods)))
axes[1].set_xticklabels(methods, fontsize=8)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("BPB Impact")
axes[1].set_title("BPB Savings: External Spellcheck vs Model-Oracle")
for i, v in enumerate(values):
    axes[1].text(
        i, v + 0.00005 * (1 if v >= 0 else -1), f"{v:+.5f}", ha="center", fontsize=8
    )

plt.tight_layout()
plt.show()

# Show model-oracle examples
sorted_oracle = sorted(oracle_results, key=lambda r: r["savings_nats"], reverse=True)
print("\n--- Top 15 model-oracle corrections that HELP ---")
for r in sorted_oracle[:15]:
    print(
        f"  '{r['actual_word']}' -> '{r['model_word']}' "
        f"savings={r['savings_nats']:.2f} nats"
    )
print("\n--- Top 10 model-oracle corrections that HURT ---")
for r in sorted_oracle[-10:]:
    print(
        f"  '{r['actual_word']}' -> '{r['model_word']}' "
        f"savings={r['savings_nats']:.2f} nats"
    )


In [ ]:
# --- Case-agnostic BPB: how much does casing cost? ---
# For each position, instead of requiring the exact token, accept any case variant.
# E.g., if target is "▁The", we sum p("▁The") + p("▁the") and use -log(sum) as loss.
# This tells us how much BPB the model spends on predicting casing.

from collections import defaultdict

# Build case-equivalence groups: tokens that share the same lowercased piece
lower_to_ids = defaultdict(list)
for tok_id in range(V):
    piece = id_to_piece[tok_id]
    lower_to_ids[piece.lower()].append(tok_id)

# For each token, store the list of all its case variants (including itself)
case_variants = {}  # token_id -> list of equivalent token_ids
for group in lower_to_ids.values():
    for tok_id in group:
        case_variants[tok_id] = group

n_with_variants = sum(1 for g in lower_to_ids.values() if len(g) > 1)
n_tokens_affected = sum(len(g) for g in lower_to_ids.values() if len(g) > 1)
print(f"Case-equivalence groups with >1 member: {n_with_variants}")
print(f"Tokens with case variants: {n_tokens_affected} / {V}")

# --- Scan validation sequences and compute case-agnostic loss ---
print(f"\nComputing case-agnostic loss over {N_BATCHES} sequences...")

original_losses = []
case_agnostic_losses = []
has_variant_mask = []  # True if this token has a case variant
original_bytes = []

for batch_idx in tqdm(range(N_BATCHES), desc="Case-agnostic scoring"):
    x, y = get_val_batch(batch_idx)
    logits = get_logits(x)  # (SEQ_LEN, V)
    log_probs = F.log_softmax(logits.float(), dim=-1)  # (SEQ_LEN, V)
    probs = log_probs.exp()

    target_ids = y[0].cpu().numpy()
    input_ids = x[0].cpu().numpy()

    for pos in range(SEQ_LEN):
        tid = int(target_ids[pos])
        prev_id = int(input_ids[pos])

        # Original loss
        orig_nll = -float(log_probs[pos, tid])

        # Case-agnostic: sum probs of all case variants
        variants = case_variants.get(tid, [tid])
        if len(variants) > 1:
            combined_prob = sum(float(probs[pos, v]) for v in variants)
            agnostic_nll = -math.log(max(combined_prob, 1e-30))
            has_var = True
        else:
            agnostic_nll = orig_nll
            has_var = False

        # Byte count (same formula as before)
        b = float(base_bytes_lut[tid])
        if has_leading_space_lut[tid] and not is_boundary_token_lut[prev_id]:
            b += 1.0

        original_losses.append(orig_nll)
        case_agnostic_losses.append(agnostic_nll)
        has_variant_mask.append(has_var)
        original_bytes.append(b)

original_losses = np.array(original_losses)
case_agnostic_losses = np.array(case_agnostic_losses)
has_variant_mask = np.array(has_variant_mask)
original_bytes = np.array(original_bytes)

# --- Compute BPB ---
total_bytes_all = original_bytes.sum()
orig_bpb = (original_losses.sum() / ln2) / total_bytes_all
agnostic_bpb = (case_agnostic_losses.sum() / ln2) / total_bytes_all
savings_bpb = orig_bpb - agnostic_bpb

# Breakdown: only on tokens that have case variants
orig_bpb_variants = (original_losses[has_variant_mask].sum() / ln2) / total_bytes_all
agnostic_bpb_variants = (
    case_agnostic_losses[has_variant_mask].sum() / ln2
) / total_bytes_all

# Per-token savings
per_token_savings = original_losses - case_agnostic_losses  # in nats, >= 0

print(f"\n{'=' * 70}")
print(f"CASE-AGNOSTIC BPB ANALYSIS")
print(f"{'=' * 70}")
print(f"Original BPB:       {orig_bpb:.6f}")
print(f"Case-agnostic BPB:  {agnostic_bpb:.6f}")
print(
    f"Savings:            {savings_bpb:.6f} BPB ({savings_bpb / orig_bpb:.2%} of total)"
)
print(f"")
print(
    f"Tokens with case variants: {has_variant_mask.sum():,} / {len(has_variant_mask):,} ({has_variant_mask.mean():.1%})"
)
print(f"BPB from variant tokens (orig):     {orig_bpb_variants:.6f}")
print(f"BPB from variant tokens (agnostic): {agnostic_bpb_variants:.6f}")
print(
    f"Savings on variant tokens only:     {orig_bpb_variants - agnostic_bpb_variants:.6f} BPB"
)

# --- Which case pairs contribute most savings? ---
# Group savings by target token
savings_by_token = defaultdict(
    lambda: {"total_savings": 0.0, "count": 0, "total_bytes": 0.0}
)
for i in range(len(original_losses)):
    if has_variant_mask[i]:
        tid = int(all_token_ids[i])
        s = savings_by_token[tid]
        s["total_savings"] += per_token_savings[i]
        s["count"] += 1
        s["total_bytes"] += original_bytes[i]

savings_df = pd.DataFrame(
    [
        {
            "token_id": tid,
            "piece": id_to_piece[tid],
            "variants": [id_to_piece[v] for v in case_variants[tid]],
            "total_savings_nats": d["total_savings"],
            "total_savings_bits": d["total_savings"] / ln2,
            "bpb_contribution": d["total_savings"] / ln2 / total_bytes_all,
            "count": d["count"],
            "mean_savings_nats": d["total_savings"] / max(d["count"], 1),
        }
        for tid, d in savings_by_token.items()
    ]
).sort_values("total_savings_nats", ascending=False)

print(f"\n--- Top 20 tokens by total case-agnostic savings ---")
print(
    savings_df.head(20)[
        ["piece", "variants", "count", "mean_savings_nats", "bpb_contribution"]
    ].to_string(index=False)
)

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1: BPB breakdown
categories = ["Total BPB", "Case-agnostic\nBPB", "Casing cost"]
values = [orig_bpb, agnostic_bpb, savings_bpb]
colors_bar = ["steelblue", "lightblue", "coral"]
axes[0].bar(categories, values, color=colors_bar, edgecolor="black", linewidth=0.5)
for i, v in enumerate(values):
    axes[0].text(i, v + 0.001, f"{v:.5f}", ha="center", fontsize=9)
axes[0].set_ylabel("BPB")
axes[0].set_title(
    f"BPB: Original vs Case-Agnostic\n(savings = {savings_bpb:.5f}, {savings_bpb / orig_bpb:.2%} of total)"
)

# 2: Distribution of per-token savings (only where has variant)
variant_savings = per_token_savings[has_variant_mask]
axes[1].hist(variant_savings, bins=80, color="coral", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Per-token savings (nats)")
axes[1].set_ylabel("Count")
axes[1].set_title(
    f"Per-token Casing Savings Distribution\n(mean={variant_savings.mean():.3f} nats, only variant tokens)"
)
axes[1].axvline(
    variant_savings.mean(),
    color="green",
    linewidth=2,
    label=f"Mean={variant_savings.mean():.3f}",
)
axes[1].legend()

# 3: Top tokens by BPB contribution
top_n = min(15, len(savings_df))
top = savings_df.head(top_n)
axes[2].barh(range(top_n), top["bpb_contribution"].values, color="coral")
axes[2].set_yticks(range(top_n))
axes[2].set_yticklabels(
    [
        f"{r['piece']!r} <-> {[v for v in r['variants'] if v != r['piece']]}"
        for _, r in top.iterrows()
    ],
    fontsize=7,
)
axes[2].set_xlabel("BPB Contribution of Casing")
axes[2].set_title("Top Tokens: BPB Spent on Case Prediction")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

# --- Where does casing cost concentrate? ---
# By position type
df_case = pd.DataFrame(
    {
        "savings_nats": per_token_savings,
        "has_variant": has_variant_mask,
        "category": df["category"].values,
        "doc_position": df["doc_position"].values,
        "bytes": original_bytes,
    }
)
df_case["is_doc_start"] = df_case["doc_position"] < 5

case_by_cat = (
    df_case[df_case["has_variant"]]
    .groupby("category")
    .agg(
        total_savings_nats=("savings_nats", "sum"),
        count=("savings_nats", "count"),
        mean_savings=("savings_nats", "mean"),
    )
)
case_by_cat["bpb_savings"] = case_by_cat["total_savings_nats"] / ln2 / total_bytes_all
case_by_cat = case_by_cat.sort_values("bpb_savings", ascending=False)
print(f"\n--- Casing cost by token category ---")
print(case_by_cat.to_string(float_format="%.6f"))

# Doc start vs body
for label, mask in [
    ("Doc start (pos<5)", df_case["is_doc_start"]),
    ("Doc body (pos>=5)", ~df_case["is_doc_start"]),
]:
    sub = df_case[mask & df_case["has_variant"]]
    bpb_sav = sub["savings_nats"].sum() / ln2 / total_bytes_all
    print(
        f"\n{label}: {bpb_sav:.6f} BPB savings ({sub['has_variant'].sum():,} variant tokens)"
    )


## 9b. Lookahead Reranking via Monte Carlo Trajectory Sampling

**Idea:** Instead of scoring token $t$ by $p(x_t \mid x_{<t})$ alone, reweight the top-$K$ candidates by how well each predicts the *actual* next $L$ tokens:

$$p_{\text{new}}(c_i) \propto p(c_i \mid x_{<t}) \cdot p(y_{t+1:t+L} \mid x_{<t},\; x_t = c_i)$$

The lookahead factor is what MCMC trajectory sampling estimates: draw $N$ trajectories autoregressively from $p(\cdot \mid c_i)$ and measure how much probability mass lands on the actual future. We compute both the **exact** lookahead (teacher-forced forward passes, equivalent to $N \to \infty$) and an **MCMC estimate** (sampled trajectories) for comparison.

In [ ]:
# --- Oracle lookahead reranking (non-causal, proper normalization) ---
# Uses actual future tokens y_{t+1:t+L} — this is an upper bound analysis.
#
# Key fix: the raw lookahead signal log p(future|c_i) is strongly negative
# (product of 3 probabilities). Using exp(raw) as f gives f << 1 for all
# tested candidates while f_default = 1, inflating the untested tail.
#
# Solution: CENTER the signal per position so mean = 0.
#   signal_i = lookahead_lp_i - mean(lookahead_lp)
#   f(c_i) = exp(α · signal_i)        for tested candidates
#   f_default = exp(0) = 1             for untested (= "average quality")
#   Z = Σ_{top-K} p(c_i)·f(c_i) + tail_mass · 1
#
# Now f_default = 1 means "this untested token has average lookahead quality
# among the candidates we tested," which is a principled uninformative prior.
import time

K_TOP = 10
LOOKAHEAD = 3
N_LA_BATCHES = 10
EVAL_STRIDE = 20
ALPHA_GRID_ORA = [0.0, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

print(f"Oracle lookahead: K={K_TOP}, L={LOOKAHEAD}, stride={EVAL_STRIDE}")
print(f"Alpha grid: {ALPHA_GRID_ORA}")
print(f"NOTE: non-causal — uses actual future tokens")

records_oracle = []
t0 = time.time()

for batch_idx in tqdm(range(N_LA_BATCHES), desc="Oracle lookahead"):
    x, y = get_val_batch(batch_idx)
    input_ids = x[0].cpu()
    target_ids = y[0].cpu()

    logits_orig = get_logits(x)
    log_probs_orig = F.log_softmax(logits_orig.float(), dim=-1)
    probs_orig = log_probs_orig.exp()

    for pos in range(0, SEQ_LEN - LOOKAHEAD, EVAL_STRIDE):
        target = target_ids[pos].item()
        orig_nll = -log_probs_orig[pos, target].item()
        target_prob = probs_orig[pos, target].item()

        topk_lp, topk_idx = log_probs_orig[pos].topk(K_TOP)
        topk_lp = topk_lp.cpu().float()
        topk_idx = topk_idx.cpu()
        topk_probs = topk_lp.exp()

        in_topk = target in topk_idx.tolist()
        target_ci = (
            (topk_idx == target).nonzero(as_tuple=True)[0].item() if in_topk else -1
        )
        sum_topk = topk_probs.sum().item()
        tail_mass = 1.0 - sum_topk

        # Lookahead: for each candidate, compute log p(actual future | candidate)
        x_batch = x.repeat(K_TOP, 1)
        for ci in range(K_TOP):
            x_batch[ci, pos + 1] = topk_idx[ci].item()

        logits_batch = get_logits_batched(x_batch)
        lp_batch = F.log_softmax(
            logits_batch[:, pos + 1 : pos + 1 + LOOKAHEAD, :].float(), dim=-1
        )

        lookahead_lps = np.zeros(K_TOP)
        for ci in range(K_TOP):
            future_lp = 0.0
            for step in range(LOOKAHEAD):
                future_lp += lp_batch[
                    ci, step, target_ids[pos + 1 + step].item()
                ].item()
            lookahead_lps[ci] = future_lp

        # Byte count
        tid = target
        prev = input_ids[pos].item()
        b = float(base_bytes_lut[tid])
        if has_leading_space_lut[tid] and not is_boundary_token_lut[prev]:
            b += 1.0

        records_oracle.append(
            {
                "topk_probs": topk_probs.numpy().copy(),
                "topk_idx": topk_idx.numpy().copy(),
                "lookahead_lps": lookahead_lps.copy(),
                "target_ci": target_ci,
                "in_topk": in_topk,
                "orig_nll": orig_nll,
                "target_prob": target_prob,
                "tail_mass": tail_mass,
                "byte_count": b,
                "pos": pos,
            }
        )

elapsed = time.time() - t0
print(f"Collection done in {elapsed:.1f}s ({len(records_oracle)} positions)")


# === Alpha sweep with centered signal ===
def sweep_alpha_centered(records, signal_key, alpha_grid, label):
    """
    Sweep alpha with CENTERED signal and proper full-vocab normalization.
    Centering ensures f_default=1 means 'average quality among tested candidates.'
    """
    results = []
    for alpha in alpha_grid:
        total_nll_orig = 0.0
        total_nll_new = 0.0
        total_bytes = 0.0
        n_helped = 0
        n_hurt = 0
        n_in_topk = 0

        for rec in records:
            topk_probs = rec["topk_probs"]
            raw_signal = rec[signal_key]
            target_ci = rec["target_ci"]
            in_topk = rec["in_topk"]
            target_prob = rec["target_prob"]
            tail_mass = rec["tail_mass"]
            b = rec["byte_count"]
            orig_nll = rec["orig_nll"]

            # Center signal: mean = 0, so exp(0) = 1 = f_default
            signal_centered = raw_signal - raw_signal.mean()

            # f(c_i) = exp(α · centered_signal_i)
            f_topk = np.exp(alpha * signal_centered)
            f_default = 1.0  # = exp(α · 0) = average quality

            # Z = Σ p(c_i)·f(c_i) + tail_mass · f_default
            Z = (topk_probs * f_topk).sum() + tail_mass * f_default

            if in_topk:
                p_new = target_prob * f_topk[target_ci] / Z
                n_in_topk += 1
            else:
                p_new = target_prob * f_default / Z

            new_nll = -math.log(max(p_new, 1e-30))
            total_nll_orig += orig_nll
            total_nll_new += new_nll
            total_bytes += b
            if new_nll < orig_nll - 0.001:
                n_helped += 1
            elif new_nll > orig_nll + 0.001:
                n_hurt += 1

        orig_bpb = (total_nll_orig / ln2) / total_bytes
        new_bpb = (total_nll_new / ln2) / total_bytes
        savings = orig_bpb - new_bpb
        results.append(
            {
                "alpha": alpha,
                "orig_bpb": orig_bpb,
                "new_bpb": new_bpb,
                "savings": savings,
                "pct": savings / orig_bpb * 100 if orig_bpb > 0 else 0,
                "n_helped": n_helped,
                "n_hurt": n_hurt,
                "n_total": len(records),
                "n_in_topk": n_in_topk,
            }
        )

    df_r = pd.DataFrame(results)
    best = df_r.loc[df_r["savings"].idxmax()]
    print(f"\n--- {label} ---")
    print(
        f"(Target in top-{K_TOP}: {results[0]['n_in_topk']}/{results[0]['n_total']} "
        f"= {results[0]['n_in_topk'] / results[0]['n_total']:.1%})"
    )
    print(
        df_r[
            ["alpha", "orig_bpb", "new_bpb", "savings", "pct", "n_helped", "n_hurt"]
        ].to_string(index=False, float_format="%.6f")
    )
    print(
        f"\nBest α={best['alpha']:.2f}: savings = {best['savings']:+.6f} BPB ({best['pct']:+.3f}%)"
    )
    return df_r, best


print(f"\n{'=' * 70}")
print(f"ORACLE LOOKAHEAD — CENTERED SIGNAL ALPHA SWEEP")
print(f"{'=' * 70}")

df_oracle, best_oracle = sweep_alpha_centered(
    records_oracle,
    "lookahead_lps",
    ALPHA_GRID_ORA,
    "Oracle lookahead (centered, non-causal)",
)

# --- Detailed analysis at best alpha ---
best_a = best_oracle["alpha"]
improvements_ora = []
in_topk_ora = []
for rec in records_oracle:
    sig = rec["lookahead_lps"] - rec["lookahead_lps"].mean()
    f_topk = np.exp(best_a * sig)
    Z = (rec["topk_probs"] * f_topk).sum() + rec["tail_mass"]
    if rec["in_topk"]:
        p_new = rec["target_prob"] * f_topk[rec["target_ci"]] / Z
    else:
        p_new = rec["target_prob"] / Z
    new_nll = -math.log(max(p_new, 1e-30))
    improvements_ora.append(rec["orig_nll"] - new_nll)
    in_topk_ora.append(rec["in_topk"])

improvements_ora = np.array(improvements_ora)
in_topk_ora = np.array(in_topk_ora)

print(f"\n--- Per-token analysis at α={best_a:.2f} ---")
print(f"  Overall mean:      {improvements_ora.mean():.6f} nats")
for label, mask in [("In top-K", in_topk_ora), ("Out of top-K", ~in_topk_ora)]:
    if mask.sum() == 0:
        continue
    imp = improvements_ora[mask]
    print(
        f"  {label:18s}: mean={imp.mean():.6f}, helped={(imp > 0.001).sum()}, "
        f"hurt={(imp < -0.001).sum()} (n={mask.sum()})"
    )

# Breakdown by loss bucket
orig_ora = np.array([r["orig_nll"] for r in records_oracle])
byte_ora = np.array([r["byte_count"] for r in records_oracle])
new_ora = orig_ora - improvements_ora

loss_bins = [0, 0.5, 1.0, 2.0, 4.0, 8.0, 100]
print(f"\n--- Savings by original loss bucket (α={best_a}) ---")
print(
    f"{'Loss range':>15s} {'Count':>8s} {'Orig BPB':>10s} {'Rerank BPB':>12s} {'Savings':>10s}"
)
for lo, hi in zip(loss_bins[:-1], loss_bins[1:]):
    mask = (orig_ora >= lo) & (orig_ora < hi)
    if mask.sum() < 10:
        continue
    o = (orig_ora[mask].sum() / ln2) / byte_ora[mask].sum()
    n = (new_ora[mask].sum() / ln2) / byte_ora[mask].sum()
    print(
        f"  [{lo:.1f}, {hi:.1f}) {mask.sum():>8d} {o:>10.4f} {n:>12.4f} {o - n:>+10.4f}"
    )

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    df_oracle["alpha"], df_oracle["savings"], "o-", color="steelblue", markersize=6
)
axes[0].axhline(0, color="black", linewidth=1, linestyle="--")
axes[0].set_xlabel("Alpha")
axes[0].set_ylabel("BPB Savings (positive = improved)")
axes[0].set_title(
    f"Oracle Lookahead: BPB Savings vs Alpha\n(centered signal, K={K_TOP}, L={LOOKAHEAD})"
)
axes[0].grid(alpha=0.3)

mask_nz = np.abs(improvements_ora) > 1e-6
if in_topk_ora.any():
    axes[1].hist(
        improvements_ora[mask_nz & in_topk_ora],
        bins=60,
        alpha=0.6,
        color="seagreen",
        label=f"In top-K (n={in_topk_ora.sum()})",
    )
if (~in_topk_ora).any():
    axes[1].hist(
        improvements_ora[mask_nz & ~in_topk_ora],
        bins=60,
        alpha=0.6,
        color="coral",
        label=f"Out of top-K (n={(~in_topk_ora).sum()})",
    )
axes[1].axvline(0, color="black", linewidth=1, linestyle="--")
axes[1].axvline(
    improvements_ora.mean(),
    color="red",
    linewidth=2,
    label=f"Mean={improvements_ora.mean():.5f}",
)
axes[1].set_xlabel("Improvement (nats)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Oracle Per-token Improvement (α={best_a})")
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / "lookahead_reranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lookahead_reranking.png")

### Fully Causal Lookahead (no future tokens used)

The oracle cell above "cheats" by conditioning on the actual future $y_{t+1:t+L}$.
Here we reweight candidates using **only the model's own predictions** — no future tokens are ever observed.

**Two causal signals:**
1. **1-step confidence:** After substituting candidate $c_i$, measure how *confident* the model is about the next token (negative entropy). Prefer candidates that reduce downstream uncertainty.
2. **MCMC trajectory log-prob:** Sample $N$ trajectories of $L$ tokens autoregressively from $p(\cdot \mid c_i)$. Score each candidate by the average joint log-probability of its sampled continuations. Candidates leading to high-probability (natural, coherent) futures are upweighted.

Reweighting: $p_{\text{new}}(c_i) \propto p(c_i) \cdot \exp\!\big(\alpha \cdot \text{signal}(c_i)\big)$ with $\alpha$ swept to find optimal.

In [ ]:
# --- Fully causal lookahead: PROPER evaluation with centered signals ---
# NO actual future tokens are ever observed or used.
#
# Signals are CENTERED per position (mean=0 across candidates) so that
# f_default = exp(0) = 1 represents "average quality among tested candidates."
# This prevents the scale mismatch where raw signal values << 0 would
# inflate the untested tail mass.
import time

K_TOP_C = 10
LOOKAHEAD_C = 3
N_CAUSAL_BATCHES = 10
EVAL_STRIDE_C = 20
MC_STRIDE_C = 100
N_TRAJ_C = 16

ALPHA_GRID = [0.0, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]

n_1step = N_CAUSAL_BATCHES * len(range(0, SEQ_LEN - LOOKAHEAD_C, EVAL_STRIDE_C))
n_mcmc = N_CAUSAL_BATCHES * len(range(0, SEQ_LEN - LOOKAHEAD_C, MC_STRIDE_C))
print(f"Fully Causal Lookahead (centered signals, proper normalization)")
print(f"  1-step: ~{n_1step} pos (K={K_TOP_C}, stride={EVAL_STRIDE_C})")
print(
    f"  MCMC:   ~{n_mcmc} pos (K={K_TOP_C}, N={N_TRAJ_C}, L={LOOKAHEAD_C}, stride={MC_STRIDE_C})"
)
print(f"  Alpha grid: {ALPHA_GRID}")

records_1step = []
records_mcmc = []
t0 = time.time()

for batch_idx in tqdm(range(N_CAUSAL_BATCHES), desc="Causal lookahead"):
    x, y = get_val_batch(batch_idx)
    input_ids = x[0].cpu()
    target_ids = y[0].cpu()

    logits_orig = get_logits(x)
    lp_orig = F.log_softmax(logits_orig.float(), dim=-1)
    probs_orig = lp_orig.exp()

    for pos in range(0, SEQ_LEN - LOOKAHEAD_C, EVAL_STRIDE_C):
        target = target_ids[pos].item()
        orig_nll = -lp_orig[pos, target].item()
        target_prob = probs_orig[pos, target].item()

        topk_lp, topk_idx = lp_orig[pos].topk(K_TOP_C)
        topk_lp = topk_lp.cpu().float()
        topk_idx = topk_idx.cpu()
        topk_probs = topk_lp.exp()

        in_topk = target in topk_idx.tolist()
        target_ci = (
            (topk_idx == target).nonzero(as_tuple=True)[0].item() if in_topk else -1
        )
        sum_topk = topk_probs.sum().item()
        tail_mass = 1.0 - sum_topk

        # Byte count
        tid = target
        prev = input_ids[pos].item()
        b = float(base_bytes_lut[tid])
        if has_leading_space_lut[tid] and not is_boundary_token_lut[prev]:
            b += 1.0

        # --- 1-step: batched forward ---
        x_batch = x.repeat(K_TOP_C, 1)
        for ci in range(K_TOP_C):
            x_batch[ci, pos + 1] = topk_idx[ci].item()

        logits_k = get_logits_batched(x_batch)
        lp_next = F.log_softmax(logits_k[:, pos + 1, :].float(), dim=-1)
        p_next = lp_next.exp()

        ent = -(p_next * lp_next).sum(dim=-1).cpu().numpy()
        neg_entropy = -ent

        records_1step.append(
            {
                "topk_probs": topk_probs.numpy().copy(),
                "topk_idx": topk_idx.numpy().copy(),
                "target_ci": target_ci,
                "in_topk": in_topk,
                "orig_nll": orig_nll,
                "target_prob": target_prob,
                "tail_mass": tail_mass,
                "byte_count": b,
                "neg_entropy": neg_entropy.copy(),
                "pos": pos,
            }
        )

        # --- MCMC: sparser grid ---
        if pos % MC_STRIDE_C != 0:
            continue

        traj_avg_lp = np.zeros(K_TOP_C)
        for ci in range(K_TOP_C):
            samples_1 = torch.multinomial(p_next[ci], N_TRAJ_C, replacement=True)
            lp_s1 = lp_next[ci, samples_1].cpu().numpy()
            traj_lps = lp_s1.copy()

            x_s2 = x.repeat(N_TRAJ_C, 1)
            x_s2[:, pos + 1] = topk_idx[ci].item()
            for n in range(N_TRAJ_C):
                x_s2[n, pos + 2] = samples_1[n].item()

            logits_s2 = get_logits_batched(x_s2)
            lp_s2_all = F.log_softmax(logits_s2[:, pos + 2, :].float(), dim=-1)
            p_s2_all = lp_s2_all.exp()
            samples_2 = torch.multinomial(p_s2_all, 1).squeeze(-1)
            lp_s2 = lp_s2_all[torch.arange(N_TRAJ_C), samples_2].cpu().numpy()
            traj_lps += lp_s2

            if pos + 3 < SEQ_LEN:
                x_s3 = x_s2.clone()
                for n in range(N_TRAJ_C):
                    x_s3[n, pos + 3] = samples_2[n].item()
                logits_s3 = get_logits_batched(x_s3)
                lp_s3_all = F.log_softmax(logits_s3[:, pos + 3, :].float(), dim=-1)
                p_s3_all = lp_s3_all.exp()
                samples_3 = torch.multinomial(p_s3_all, 1).squeeze(-1)
                lp_s3 = lp_s3_all[torch.arange(N_TRAJ_C), samples_3].cpu().numpy()
                traj_lps += lp_s3

            traj_avg_lp[ci] = traj_lps.mean()

        records_mcmc.append(
            {
                "topk_probs": topk_probs.numpy().copy(),
                "topk_idx": topk_idx.numpy().copy(),
                "target_ci": target_ci,
                "in_topk": in_topk,
                "orig_nll": orig_nll,
                "target_prob": target_prob,
                "tail_mass": tail_mass,
                "byte_count": b,
                "traj_avg_lp": traj_avg_lp.copy(),
                "pos": pos,
            }
        )

elapsed = time.time() - t0
print(f"\nCollection done in {elapsed:.1f}s")
print(f"  1-step records: {len(records_1step):,}")
print(f"  MCMC records:   {len(records_mcmc):,}")


# === Alpha sweep with centered signal and proper normalization ===
# (same function used by oracle cell — defined here if oracle cell wasn't run)
def sweep_alpha_centered(records, signal_key, alpha_grid, label):
    results = []
    for alpha in alpha_grid:
        total_nll_orig = 0.0
        total_nll_new = 0.0
        total_bytes = 0.0
        n_helped = 0
        n_hurt = 0
        n_in_topk = 0

        for rec in records:
            topk_probs = rec["topk_probs"]
            raw_signal = rec[signal_key]
            signal_centered = raw_signal - raw_signal.mean()

            f_topk = np.exp(alpha * signal_centered)
            f_default = 1.0
            Z = (topk_probs * f_topk).sum() + rec["tail_mass"] * f_default

            if rec["in_topk"]:
                p_new = rec["target_prob"] * f_topk[rec["target_ci"]] / Z
                n_in_topk += 1
            else:
                p_new = rec["target_prob"] * f_default / Z

            new_nll = -math.log(max(p_new, 1e-30))
            total_nll_orig += rec["orig_nll"]
            total_nll_new += new_nll
            total_bytes += rec["byte_count"]
            if new_nll < rec["orig_nll"] - 0.001:
                n_helped += 1
            elif new_nll > rec["orig_nll"] + 0.001:
                n_hurt += 1

        orig_bpb = (total_nll_orig / ln2) / total_bytes
        new_bpb = (total_nll_new / ln2) / total_bytes
        savings = orig_bpb - new_bpb
        results.append(
            {
                "alpha": alpha,
                "orig_bpb": orig_bpb,
                "new_bpb": new_bpb,
                "savings": savings,
                "pct": savings / orig_bpb * 100 if orig_bpb > 0 else 0,
                "n_helped": n_helped,
                "n_hurt": n_hurt,
                "n_total": len(records),
                "n_in_topk": n_in_topk,
            }
        )

    df_r = pd.DataFrame(results)
    best = df_r.loc[df_r["savings"].idxmax()]
    print(f"\n--- {label} ---")
    print(
        f"(Target in top-{K_TOP_C}: {results[0]['n_in_topk']}/{results[0]['n_total']} "
        f"= {results[0]['n_in_topk'] / results[0]['n_total']:.1%})"
    )
    print(
        df_r[
            ["alpha", "orig_bpb", "new_bpb", "savings", "pct", "n_helped", "n_hurt"]
        ].to_string(index=False, float_format="%.6f")
    )
    print(
        f"\nBest α={best['alpha']:.2f}: {best['savings']:+.6f} BPB ({best['pct']:+.3f}%)"
    )
    return df_r, best


print(f"\n{'=' * 70}")
print(f"CAUSAL LOOKAHEAD — CENTERED SIGNAL ALPHA SWEEP")
print(f"{'=' * 70}")

df_1step, best_1step = sweep_alpha_centered(
    records_1step, "neg_entropy", ALPHA_GRID, "1-step confidence (neg entropy)"
)

df_mcmc, best_mcmc = sweep_alpha_centered(
    records_mcmc, "traj_avg_lp", ALPHA_GRID, "MCMC trajectory log-prob"
)

# === Analysis at best alpha ===
best_a1 = best_1step["alpha"]
best_am = best_mcmc["alpha"]

improvements_1s = []
in_topk_1s = []
for rec in records_1step:
    sig = rec["neg_entropy"] - rec["neg_entropy"].mean()
    f = np.exp(best_a1 * sig)
    Z = (rec["topk_probs"] * f).sum() + rec["tail_mass"]
    if rec["in_topk"]:
        p_new = rec["target_prob"] * f[rec["target_ci"]] / Z
    else:
        p_new = rec["target_prob"] / Z
    improvements_1s.append(rec["orig_nll"] - (-math.log(max(p_new, 1e-30))))
    in_topk_1s.append(rec["in_topk"])

improvements_1s = np.array(improvements_1s)
in_topk_1s = np.array(in_topk_1s)

print(f"\n--- 1-step @ α={best_a1}: per-token ---")
print(f"  Mean improvement: {improvements_1s.mean():.6f} nats")
for lbl, msk in [("In top-K", in_topk_1s), ("Out of top-K", ~in_topk_1s)]:
    if msk.sum() == 0:
        continue
    i = improvements_1s[msk]
    print(
        f"  {lbl:18s}: mean={i.mean():.6f}, helped={(i > 0.001).sum()}, hurt={(i < -0.001).sum()} (n={msk.sum()})"
    )

# Loss bucket breakdown
orig_cs = np.array([r["orig_nll"] for r in records_1step])
byte_cs = np.array([r["byte_count"] for r in records_1step])
new_cs = orig_cs - improvements_1s

loss_bins = [0, 0.5, 1.0, 2.0, 4.0, 8.0, 100]
print(f"\n--- 1-step savings by loss bucket (α={best_a1}) ---")
print(
    f"{'Loss range':>15s} {'Count':>8s} {'Orig BPB':>10s} {'New BPB':>10s} {'Savings':>10s}"
)
for lo, hi in zip(loss_bins[:-1], loss_bins[1:]):
    mask = (orig_cs >= lo) & (orig_cs < hi)
    if mask.sum() < 10:
        continue
    o = (orig_cs[mask].sum() / ln2) / byte_cs[mask].sum()
    n = (new_cs[mask].sum() / ln2) / byte_cs[mask].sum()
    print(
        f"  [{lo:.1f}, {hi:.1f}) {mask.sum():>8d} {o:>10.4f} {n:>10.4f} {o - n:>+10.4f}"
    )

# === Plots ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1: Alpha sweep curves
axes[0].plot(
    df_1step["alpha"],
    df_1step["savings"],
    "o-",
    color="steelblue",
    label="1-step confidence",
    markersize=5,
)
axes[0].plot(
    df_mcmc["alpha"],
    df_mcmc["savings"],
    "s-",
    color="coral",
    label="MCMC trajectory",
    markersize=5,
)
axes[0].axhline(0, color="black", linewidth=1, linestyle="--")
axes[0].set_xlabel("Alpha")
axes[0].set_ylabel("BPB Savings")
axes[0].set_title(
    "Causal Lookahead: BPB Savings vs Alpha\n(centered signals, full-vocab norm)"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2: Per-token improvement
mask_nz = np.abs(improvements_1s) > 1e-6
if mask_nz.any():
    axes[1].hist(
        improvements_1s[mask_nz],
        bins=100,
        color="steelblue",
        alpha=0.7,
        edgecolor="black",
    )
    axes[1].axvline(0, color="black", linewidth=1, linestyle="--")
    axes[1].axvline(
        improvements_1s.mean(),
        color="red",
        linewidth=2,
        label=f"Mean={improvements_1s.mean():.6f}",
    )
    axes[1].set_xlabel("Improvement (nats)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"1-step (α={best_a1}): Per-token")
    axes[1].legend()

# 3: Comparison with oracle
try:
    comp_labels = [
        "Original",
        f"Causal 1-step\nα={best_a1}",
        f"Causal MCMC\nα={best_am}",
        f"Oracle\nα={best_oracle['alpha']:.1f}",
    ]
    comp_bpb = [
        best_1step["orig_bpb"],
        best_1step["new_bpb"],
        best_mcmc["new_bpb"],
        best_oracle["new_bpb"],
    ]
    comp_colors = ["steelblue", "seagreen", "coral", "gold"]
except NameError:
    comp_labels = [
        "Original",
        f"Causal 1-step\nα={best_a1}",
        f"Causal MCMC\nα={best_am}",
    ]
    comp_bpb = [best_1step["orig_bpb"], best_1step["new_bpb"], best_mcmc["new_bpb"]]
    comp_colors = ["steelblue", "seagreen", "coral"]

bars = axes[2].bar(
    comp_labels, comp_bpb, color=comp_colors, edgecolor="black", linewidth=0.5
)
for i, v in enumerate(comp_bpb):
    axes[2].text(i, v + max(comp_bpb) * 0.005, f"{v:.5f}", ha="center", fontsize=8)
axes[2].set_ylabel("BPB")
axes[2].set_title("BPB Comparison (all properly normalized)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "causal_lookahead_reranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: causal_lookahead_reranking.png")

print(f"\n{'=' * 70}")
print(f"SUMMARY (proper centered signals, full-vocab normalization)")
print(f"{'=' * 70}")
print(f"Original BPB:                    {best_1step['orig_bpb']:.6f}")
print(
    f"Causal 1-step (best α={best_a1:.2f}):     {best_1step['new_bpb']:.6f}  ({best_1step['savings']:+.6f})"
)
print(
    f"Causal MCMC   (best α={best_am:.2f}):     {best_mcmc['new_bpb']:.6f}  ({best_mcmc['savings']:+.6f})"
)
try:
    print(
        f"Oracle        (best α={best_oracle['alpha']:.2f}):     {best_oracle['new_bpb']:.6f}  ({best_oracle['savings']:+.6f})"
    )
except NameError:
    pass

In [ ]:
# --- Top-K truncation + Temperature sweep ---
# Two orthogonal distribution modifications, evaluated properly:
# 1. Temperature: logits / T before softmax (T<1 sharpens, T>1 flattens)
# 2. Top-K truncation: keep top-K probs, floor the rest, renormalize
# Both are free — no extra forward passes, just post-hoc logit manipulation.
import time

K_GRID = [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, V]
T_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.5, 2.0]
P_FLOOR = 1e-10

print(f"Top-K + Temperature sweep over full validation set")
print(f"K values: {K_GRID}")
print(f"T values: {T_GRID}")
print(f"Floor: {P_FLOOR}")

# Storage: results_k[k] for top-K sweep (at T=1), results_t[t] for temperature sweep (full vocab)
results_k = {
    k: {
        "nll_orig": 0.0,
        "nll_trunc": 0.0,
        "total_bytes": 0.0,
        "in_topk": 0,
        "n_total": 0,
    }
    for k in K_GRID
}
results_t = {
    t: {"nll_orig": 0.0, "nll_scaled": 0.0, "total_bytes": 0.0, "n_total": 0}
    for t in T_GRID
}

t0 = time.time()

for batch_idx in tqdm(range(N_BATCHES), desc="K + T sweep"):
    x, y = get_val_batch(batch_idx)
    input_ids = x[0].cpu()
    target_ids = y[0].cpu()

    logits = get_logits(x)  # (SEQ_LEN, V) — raw logits

    # Precompute temperature-scaled log probs for all T values
    lp_by_t = {}
    for t in T_GRID:
        lp_by_t[t] = F.log_softmax(logits.float() / t, dim=-1)  # (SEQ_LEN, V)

    # T=1 probs for top-K sweep
    log_probs = lp_by_t[1.0]
    probs = log_probs.exp()

    for pos in range(SEQ_LEN):
        tid = target_ids[pos].item()
        prev = input_ids[pos].item()
        b = float(base_bytes_lut[tid])
        if has_leading_space_lut[tid] and not is_boundary_token_lut[prev]:
            b += 1.0

        orig_nll = -log_probs[pos, tid].item()

        # --- Temperature sweep (full vocab, no truncation) ---
        for t in T_GRID:
            r = results_t[t]
            r["n_total"] += 1
            r["total_bytes"] += b
            r["nll_orig"] += orig_nll
            r["nll_scaled"] += -lp_by_t[t][pos, tid].item()

        # --- Top-K sweep (T=1) ---
        sorted_probs, sorted_idx = probs[pos].sort(descending=True)
        target_rank = (sorted_idx == tid).nonzero(as_tuple=True)[0].item()
        target_prob = probs[pos, tid].item()

        for k in K_GRID:
            r = results_k[k]
            r["n_total"] += 1
            r["total_bytes"] += b
            r["nll_orig"] += orig_nll

            if k >= V:
                r["nll_trunc"] += orig_nll
                r["in_topk"] += 1
                continue

            sum_topk = sorted_probs[:k].sum().item()
            Z = sum_topk + (V - k) * P_FLOOR

            if target_rank < k:
                trunc_nll = -math.log(target_prob / Z)
                r["in_topk"] += 1
            else:
                trunc_nll = -math.log(P_FLOOR / Z)

            r["nll_trunc"] += trunc_nll

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s\n")

# =============================================
# Temperature results
# =============================================
print(f"{'=' * 70}")
print(f"TEMPERATURE SCALING (full vocab)")
print(f"{'=' * 70}")
print(
    f"{'T':>6s} {'Orig BPB':>10s} {'Scaled BPB':>11s} {'Savings':>10s} {'Relative':>10s}"
)
print("-" * 52)

t_rows = []
for t in T_GRID:
    r = results_t[t]
    orig_bpb = (r["nll_orig"] / ln2) / r["total_bytes"]
    scaled_bpb = (r["nll_scaled"] / ln2) / r["total_bytes"]
    savings = orig_bpb - scaled_bpb
    pct = savings / orig_bpb * 100
    marker = (
        " <-- best"
        if t != 1.0
        and abs(savings)
        == max(
            abs(
                (results_t[tt]["nll_orig"] - results_t[tt]["nll_scaled"])
                / ln2
                / results_t[tt]["total_bytes"]
            )
            for tt in T_GRID
            if tt != 1.0
        )
        else ""
    )
    print(
        f"{t:>6.2f} {orig_bpb:>10.6f} {scaled_bpb:>11.6f} {savings:>+10.6f} {pct:>+9.3f}%{marker}"
    )
    t_rows.append(
        {
            "T": t,
            "orig_bpb": orig_bpb,
            "scaled_bpb": scaled_bpb,
            "savings": savings,
            "pct": pct,
        }
    )

best_t = max(t_rows, key=lambda r: r["savings"])
print(
    f"\nBest temperature: T={best_t['T']:.2f} → {best_t['scaled_bpb']:.6f} BPB "
    f"(savings: {best_t['savings']:+.6f}, {best_t['pct']:+.3f}%)"
)

# =============================================
# Top-K results
# =============================================
print(f"\n{'=' * 70}")
print(f"TOP-K TRUNCATION (T=1, floor={P_FLOOR})")
print(f"{'=' * 70}")
print(
    f"{'K':>6s} {'Tgt in top-K':>14s} {'Orig BPB':>10s} {'Trunc BPB':>10s} "
    f"{'Savings':>10s} {'Relative':>10s}"
)
print("-" * 74)

k_rows = []
for k in K_GRID:
    r = results_k[k]
    orig_bpb = (r["nll_orig"] / ln2) / r["total_bytes"]
    trunc_bpb = (r["nll_trunc"] / ln2) / r["total_bytes"]
    savings = orig_bpb - trunc_bpb
    pct = savings / orig_bpb * 100
    in_topk_pct = r["in_topk"] / r["n_total"] * 100
    k_label = f"{k}" if k < V else "full"
    print(
        f"{k_label:>6s} {in_topk_pct:>11.2f}%   {orig_bpb:>10.6f} {trunc_bpb:>10.6f} "
        f"{savings:>+10.6f} {pct:>+9.3f}%"
    )
    k_rows.append(
        {
            "K": k,
            "orig_bpb": orig_bpb,
            "trunc_bpb": trunc_bpb,
            "savings": savings,
            "pct": pct,
            "in_topk_pct": in_topk_pct,
        }
    )

# Break-even K
for r in k_rows:
    if r["savings"] >= -0.0001 and r["K"] < V:
        print(f"\nBreak-even: ~K={r['K']}")
        break
else:
    print(f"\nTruncation hurts BPB for all K < V")

# =============================================
# Plots
# =============================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1: Temperature BPB curve
ts = [r["T"] for r in t_rows]
scaled_bpbs = [r["scaled_bpb"] for r in t_rows]
axes[0].plot(ts, scaled_bpbs, "o-", color="steelblue", markersize=6)
axes[0].axvline(1.0, color="gray", linewidth=0.5, linestyle=":")
axes[0].axhline(
    t_rows[ts.index(1.0)]["orig_bpb"],
    color="black",
    linewidth=1,
    linestyle="--",
    label=f"T=1.0 (baseline)",
)
axes[0].plot(
    best_t["T"],
    best_t["scaled_bpb"],
    "*",
    color="red",
    markersize=15,
    label=f"Best: T={best_t['T']:.2f}",
)
axes[0].set_xlabel("Temperature")
axes[0].set_ylabel("BPB")
axes[0].set_title("Temperature Scaling: BPB vs T")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2: Top-K truncation BPB
ks = [r["K"] for r in k_rows]
trunc_bpbs = [r["trunc_bpb"] for r in k_rows]
axes[1].plot(ks, trunc_bpbs, "o-", color="coral", markersize=6)
axes[1].axhline(
    k_rows[-1]["orig_bpb"],
    color="black",
    linewidth=1,
    linestyle="--",
    label=f"Full vocab = {k_rows[-1]['orig_bpb']:.4f}",
)
axes[1].set_xscale("log")
axes[1].set_xlabel("K (tokens kept)")
axes[1].set_ylabel("BPB")
axes[1].set_title(f"Top-K Truncation BPB (floor={P_FLOOR})")
axes[1].legend()
axes[1].grid(alpha=0.3)
for r in k_rows:
    if r["K"] in [1, 10, 100, 500]:
        axes[1].annotate(
            f"K={r['K']}\n{r['trunc_bpb']:.3f}",
            (r["K"], r["trunc_bpb"]),
            textcoords="offset points",
            xytext=(10, 0),
            fontsize=7,
        )

# 3: Combined view — savings for both methods
ax3 = axes[2]
# Temperature savings
t_savings = [r["savings"] for r in t_rows]
ax3.plot(ts, t_savings, "o-", color="steelblue", markersize=5, label="Temperature")
ax3.set_xlabel("Temperature / K")
ax3.set_ylabel("BPB Savings (positive = improved)")
ax3.axhline(0, color="black", linewidth=1)
ax3.legend(loc="upper left")
ax3.set_title("BPB Savings: Temperature vs Top-K")
ax3.grid(alpha=0.3)

# Add top-K on secondary x-axis
ax3b = ax3.twiny()
k_savings = [r["savings"] for r in k_rows]
ax3b.plot(ks, k_savings, "s-", color="coral", markersize=5, label="Top-K")
ax3b.set_xscale("log")
ax3b.set_xlabel("K (top-K)", color="coral")
ax3b.tick_params(axis="x", labelcolor="coral")
ax3b.legend(loc="upper right")

plt.tight_layout()
plt.savefig(PLOT_DIR / "topk_temperature_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: topk_temperature_sweep.png")

## 9. Actionable BPB Decomposition

Decompose total BPB into mutually exclusive categories and rank improvement opportunities.

In [ ]:
# --- Additive BPB decomposition by category ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Waterfall chart
cats_sorted = cat_stats.index.tolist()
contributions = cat_stats["bpb_contribution"].values
colors_wf = plt.cm.Set2(np.linspace(0, 1, len(cats_sorted)))

cumulative = 0
for i, (cat, contrib) in enumerate(zip(cats_sorted, contributions)):
    axes[0].bar(
        i,
        contrib,
        bottom=cumulative,
        color=colors_wf[i],
        edgecolor="black",
        linewidth=0.5,
    )
    axes[0].text(
        i,
        cumulative + contrib / 2,
        f"{contrib:.4f}",
        ha="center",
        va="center",
        fontsize=7,
    )
    cumulative += contrib

axes[0].set_xticks(range(len(cats_sorted)))
axes[0].set_xticklabels(cats_sorted, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("BPB")
axes[0].set_title(f"BPB Waterfall by Category (total={overall_bpb:.4f})")
axes[0].axhline(overall_bpb, color="red", linestyle="--", alpha=0.5)

# Pie chart for proportions
axes[1].pie(
    contributions,
    labels=[f"{c}\n{v:.4f}" for c, v in zip(cats_sorted, contributions)],
    colors=colors_wf,
    autopct="%1.1f%%",
    pctdistance=0.8,
    textprops={"fontsize": 7},
)
axes[1].set_title("BPB Proportion by Category")

plt.tight_layout()
plt.show()

In [ ]:
# --- Cross-cut decomposition: (doc_start vs body) x (word_start vs internal) x (rare vs common) ---
median_freq = df["train_freq"].median()

df["pos_type"] = np.where(df["doc_position"] < 5, "doc_start", "doc_body")
df["word_type"] = np.where(df["has_space"], "word_start", "word_internal")
df["freq_type"] = np.where(df["train_freq"] >= median_freq, "common", "rare")

cross_cut = (
    df.groupby(["pos_type", "word_type", "freq_type"])
    .agg(
        total_loss=("loss", "sum"),
        total_bytes=("bytes", "sum"),
        count=("loss", "count"),
        mean_loss=("loss", "mean"),
    )
    .reset_index()
)
cross_cut["bpb_contribution"] = (cross_cut["total_loss"] / ln2) / total_bytes
cross_cut["internal_bpb"] = (cross_cut["total_loss"] / ln2) / cross_cut["total_bytes"]
cross_cut["frac_bpb"] = cross_cut["bpb_contribution"] / overall_bpb
cross_cut = cross_cut.sort_values("bpb_contribution", ascending=False)

print("=== Cross-cut BPB Decomposition ===")
print(
    cross_cut[
        [
            "pos_type",
            "word_type",
            "freq_type",
            "count",
            "mean_loss",
            "internal_bpb",
            "bpb_contribution",
            "frac_bpb",
        ]
    ].to_string(index=False, float_format="%.4f")
)

# Visualize as grouped bar
fig, ax = plt.subplots(figsize=(14, 5))
labels = [
    f"{r['pos_type']}\n{r['word_type']}\n{r['freq_type']}"
    for _, r in cross_cut.iterrows()
]
colors_cc = ["coral" if "doc_start" in l else "steelblue" for l in labels]
ax.bar(
    range(len(cross_cut)),
    cross_cut["bpb_contribution"],
    color=colors_cc,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_xticks(range(len(cross_cut)))
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("BPB Contribution")
ax.set_title("Cross-cut BPB Decomposition")
for i, (_, row) in enumerate(cross_cut.iterrows()):
    ax.text(
        i,
        row["bpb_contribution"] + 0.002,
        f"{row['frac_bpb']:.1%}",
        ha="center",
        fontsize=7,
    )
plt.tight_layout()
plt.show()

In [ ]:
# --- Improvement opportunity ranking ---
# For each category, compute: current BPB, best-case internal BPB, savings potential
min_internal_bpb = cat_stats["actual_bpb"].min()

ranking = cat_stats[
    ["actual_bpb", "bpb_contribution", "frac_bpb", "frac_tokens", "frac_bytes"]
].copy()
ranking["savings_if_matched_best"] = ranking["bpb_contribution"] * (
    1 - min_internal_bpb / ranking["actual_bpb"]
)

# Add n-gram gap if available
if df_ngram is not None:
    ngram_gap_by_cat = df_ngram.groupby("category")["neural_advantage"].mean()
    ranking["ngram_advantage"] = ngram_gap_by_cat
    ranking["ngram_advantage"] = ranking["ngram_advantage"].fillna(0)

ranking = ranking.sort_values("bpb_contribution", ascending=False)

print("=" * 90)
print("IMPROVEMENT OPPORTUNITY RANKING")
print("=" * 90)
print(ranking.to_string(float_format="%.4f"))

print("\n" + "=" * 90)
print("SUMMARY: TOP IMPROVEMENT OPPORTUNITIES")
print("=" * 90)
for i, (cat, row) in enumerate(ranking.head(5).iterrows()):
    print(f"\n{i + 1}. {cat}")
    print(
        f"   BPB contribution: {row['bpb_contribution']:.4f} ({row['frac_bpb']:.1%} of total)"
    )
    print(f"   Internal BPB: {row['actual_bpb']:.4f}")
    print(
        f"   Token fraction: {row['frac_tokens']:.1%}, Byte fraction: {row['frac_bytes']:.1%}"
    )
    print(
        f"   Max savings (if matched best category): {row['savings_if_matched_best']:.4f} BPB"
    )
    if df_ngram is not None and "ngram_advantage" in ranking.columns:
        print(
            f"   Neural advantage over KenLM: {row.get('ngram_advantage', 0):.3f} nats"
        )